<a href="https://colab.research.google.com/github/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/blob/main/Coding/01_dataset_prep_and_custom_edgecases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP9800 MSc Project: Dataset Preparation & Controlled Experimental Corpus
**Project Title:** Developing Techniques for Prompt Assembly for Healthcare Applications  
**Author:** Faraz | **Supervisor:** Dr. Jinan Fiaidhi  
**Institution:** Lakehead University (F/W 2026–2027)

---

## 1. Overview & Architectural Purpose
This notebook implements **Stage 1** of our empirical evaluation pipeline: constructing and standardizing the **Tri-Corpus Dataset Architecture**.

To rigorously evaluate how modular prompt assembly (Few-Shot Exemplars, Linguistic Shifting, Negative Guardrails, and PICO-Structured Chain-of-Thought) impacts Small Language Models (IBM Granite 2B/8B), we decouple our data into three distinct roles:
1. **Primary Test Benchmark (`MTS-Dialog` / MEDIQA-Chat):** 1,700 authentic clinician-authored dialogue-note pairs used to establish baseline NLP comparability.
2. **Dynamic Few-Shot Retrieval Pool (`MedSynth` / Dial2Note):** 10,000+ synthetic dialogue-SOAP pairs across 2,000+ ICD-10 codes used strictly as an external knowledge base to retrieve 2–3 condition-matched exemplars during prompt assembly (preventing data leakage).
3. **Controlled Experimental Stress-Test Corpus (`Custom Edge-Case Set`, $N=50$):** Public datasets rarely annotate *what is intentionally missing* from a conversation. Per supervisor guidance, this custom 50-case dataset is specifically engineered to stress-test:
   * **Noise Filtering:** Stripping casual opening banter and local small talk.
   * **Linguistic Shifting:** Translating informal patient idioms into formal clinical taxonomy.
   * **Negative Guardrails:** Enforcing `"Not Documented"` outputs when critical variables (e.g., allergies, specific vitals, family history) are deliberately omitted.
   * **PICO Clinical Reasoning:** Extracting structured **[P]**atient/Diagnosis, **[I]**ntervention, **[C]**omparison/Alternative, and **[O]**utcome logic prior to drafting the Assessment and Plan.

## 2. Environment Setup & Cross-Platform Compatibility
While our primary model inference runs on **Kaggle's Dual NVIDIA T4 GPU environment** (32 GB VRAM), project deliverables require seamless execution on **Google Colab**.

The code cell below installs the required HuggingFace `datasets` and `parquet` engines and automatically detects the runtime environment (`/kaggle/working` vs. Google Colab/Local filesystem) so file paths never break when shared with the supervisor.

In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP & CROSS-PLATFORM PATH DETECTION
# Purpose: Ensure seamless portability between Kaggle (Dual T4 GPUs) and
#          Google Colab (Supervisor grading environment).
# ==============================================================================

# Install required data handling libraries quietly
!pip install -q datasets pandas pyarrow

import os
import json
import pandas as pd
from datasets import load_dataset

# 1. Detect whether the notebook is currently executing on Kaggle or Google Colab
if os.path.exists("/kaggle/working"):
    OUTPUT_DIR = "/kaggle/working/processed_data"
    ENV_NAME = "Kaggle GPU Environment (/kaggle/working)"
else:
    OUTPUT_DIR = "./processed_data"
    ENV_NAME = "Google Colab / Local Environment (./processed_data)"

# 2. Create the standardized output directory for downstream notebooks (02, 03, 04)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print(f"[STATUS] Environment Detected : {ENV_NAME}")
print(f"[STATUS] Data Output Directory: {OUTPUT_DIR}")
print("=" * 70)

## 3. Public Corpora Ingestion (`MTS-Dialog` & `MedSynth`)
In this section, we fetch our two public datasets from the HuggingFace Hub and standardize their column schemas into fast-loading Apache Parquet files:
* **`MTS-Dialog` (`har1/MTS_Dialogue-Clinical_Note`):** Serves as our primary external benchmark. Contains conversational snippets paired with clinician reference text.
* **`MedSynth` (`AhmadMianroodi/MedSynth`):** Serves exclusively as our retrieval pool for Dynamic Few-Shot Prompting. By storing this separately, we ensure zero circularity bias or test-set contamination during evaluation.

In [ ]:
# ==============================================================================
# DOWNLOAD & STANDARDIZE PUBLIC CORPORA
# Purpose: Fetch MTS-Dialog (Evaluation Benchmark) and MedSynth / SOAP-Dialog
#          (Few-Shot Pool) and save both to /kaggle/working/processed_data
# ==============================================================================

print("[1/2] Fetching MTS-Dialog (Primary Test Benchmark) from HuggingFace...")
mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note")
df_mts = pd.DataFrame(mts_dataset["train"])
df_mts.columns = [col.strip() for col in df_mts.columns]
mts_path = os.path.join(OUTPUT_DIR, "mts_dialog_benchmark.parquet")
df_mts.to_parquet(mts_path, index=False)
print(f"  -> Success! Saved {len(df_mts)} MTS-Dialog records to {mts_path}")

print("\n[2/2] Fetching MedSynth / SOAP-Dialogue (Dynamic Few-Shot Pool) from HuggingFace...")
CANDIDATE_REPOS = [
    "RaiseLab/MedSynth",
    "mianroodi/MedSynth",
    "omi-health/medical-dialogue-to-soap-summary"
]

df_medsynth = None
for repo_id in CANDIDATE_REPOS:
    try:
        print(f"  -> Trying HuggingFace repository: '{repo_id}'...")
        ds = load_dataset(repo_id, split="train")
        df_medsynth = pd.DataFrame(ds)
        print(f"  -> Success! Loaded {len(df_medsynth)} records from '{repo_id}'.")
        break
    except Exception:
        continue

if df_medsynth is not None:
    df_medsynth.columns = [col.strip() for col in df_medsynth.columns]
    # Standardize column names to ['Dialogue', 'Note'] if needed
    rename_map = {}
    for col in df_medsynth.columns:
        if col.lower() == "dialogue":
            rename_map[col] = "Dialogue"
        elif col.lower() in ["note", "soap", "summary"]:
            rename_map[col] = "Note"
    df_medsynth = df_medsynth.rename(columns=rename_map)

    medsynth_path = os.path.join(OUTPUT_DIR, "medsynth_fewshot_pool.parquet")
    df_medsynth.to_parquet(medsynth_path, index=False)
    print(f"  -> Success! Saved Few-Shot Pool to: {medsynth_path}")
    print(f"  -> Columns: {list(df_medsynth.columns)}")

## 4. Constructing the 50-Case Controlled Experimental Dataset (`ChatGPT + Personal`)
To evaluate our three-tier metric framework with 100% mathematical precision, we construct **50 controlled clinical dialogues (`EXP_001` to `EXP_050`)** spanning **10 medical specialties** (Cardiology, Gastroenterology, Pulmonology, Neurology, Endocrinology, Orthopedics, Dermatology, Psychiatry, Infectious Disease, and Nephrology/Urology).

### Why This Custom Dataset is Essential for the IEEE Paper:
1. **Explicit Ground-Truth for Negative Guardrails (Tier 3):** Every single dialogue deliberately omits 3 to 4 standard clinical parameters (recorded in `omitted_fields`, such as `Allergies`, `Family History`, `Temperature`, or `Smoking History`). Our evaluation engine checks whether the LLM properly marks these exact fields as `"Not Documented"` or `"Unknown"` rather than hallucinating normal findings.
2. **Explicit Ground-Truth for Linguistic Shifting (Tier 1 & 2):** Every dialogue embeds colloquial patient idioms mapped directly to formal medical terminology in `linguistic_shifts` (e.g., *"heart is fluttering"* $\rightarrow$ *"intermittent palpitations"*).
3. **Explicit Ground-Truth for PICO Reasoning:** In every dialogue, the patient explicitly asks about a competing or alternative treatment (e.g., OTC remedies, lifestyle monotherapy, or conservative wait-and-see), allowing us to verify whether the Prompt Assembly pipeline accurately extracts the **P-I-C-O** schema (`Patient/Diagnosis`, `Intervention`, `Comparison/Alternative`, `Outcome`).
4. **Conversational Noise Removal:** Each transcript begins with realistic doctor-patient small talk (including local Northern Ontario / Thunder Bay cultural references) that must be completely stripped from the final SOAP note.

In [ ]:
# ==============================================================================
# GENERATE THE 50-CASE CONTROLLED EXPERIMENTAL STRESS-TEST DATASET
# Purpose: Programmatically construct 50 unique, clinically rich dialogue-SOAP
#          pairs across 10 specialties with exact ground-truth annotations for:
#          (1) Small Talk Noise, (2) Linguistic Shifts, (3) Omitted Fields
#          (Negative Guardrails), and (4) PICO Clinical Reasoning.
# ==============================================================================

# 1. Define 50 distinct realistic opening small-talk exchanges (Northern Ontario / General clinical banter)
#    Condition A (Zero-Shot) often leaks these details; Condition B (Prompt Assembly) must strip them.
SMALL_TALK_POOL = [
    ("Morning Arthur! How did your granddaughter's hockey tournament go in Thunder Bay last weekend?", "Oh, it was freezing in the rink, Doc, absolute nail-biter, they lost in overtime!", "Granddaughter's hockey tournament in Thunder Bay losing in overtime."),
    ("Hi Sarah, sorry for the wait, the clinic parking lot was completely snowed in this morning.", "Tell me about it, my Civic almost got stuck in the drift by the entrance!", "Snowed-in clinic parking lot and Honda Civic getting stuck."),
    ("Hey Marcus, good to see you. Did you end up adopting that golden retriever puppy you showed me last time?", "We did! Named him Barnaby, he's chewing on every shoe in the house.", "Adopting a golden retriever puppy named Barnaby."),
    ("Hi Elena! How are midterms going over at Lakehead University this week?", "Brutal, honestly. Three computer science exams back-to-back.", "Lakehead University midterms and three exams back-to-back."),
    ("Afternoon David. Did you manage to get your boat out on Lake Superior before the weather turned?", "Got one last fishing trip in near Sleeping Giant! Caught two lake trout.", "Fishing trip near Sleeping Giant on Lake Superior."),
    ("Morning Priya! How was the drive down Highway 11/17 today?", "Pretty icy past Nipigon, had to drive forty the whole way behind a snowplow.", "Icy drive down Highway 11/17 past Nipigon behind a snowplow."),
    ("Hi Tom, how are the renovations on the basement coming along?", "Slowly, Doc. Drywall dust is everywhere and the plumber rescheduled twice.", "Basement renovations, drywall dust, and plumber rescheduling."),
    ("Good morning Linda! Did you make it out to the Kakabeka Falls farmers' market on Saturday?", "I did! Picked up some fresh maple syrup and artisan sourdough bread.", "Visiting Kakabeka Falls farmers' market for maple syrup and bread."),
    ("Hey Kevin, congrats to your son on graduating from the engineering program!", "Thanks Doc, we're super proud—he already lined up a job in Toronto next month.", "Son graduating from engineering and moving to Toronto."),
    ("Hi Chloe, how was your trip to Winnipeg to visit your sister?", "Great visit, though my flight back got delayed four hours at the airport.", "Trip to Winnipeg and a four-hour flight delay."),
    ("Morning Walter! Are you still volunteering at the curling club on Thursdays?", "Every Thursday night! Though they have me keeping score more than sweeping now.", "Volunteering and keeping score at the curling club."),
    ("Hi Nadia, did your backyard garden survive that early frost we had on Tuesday?", "Covered the tomatoes with bedsheets just in time, saved most of them!", "Covering backyard tomato plants to survive early frost."),
    ("Afternoon Greg! How did the smoked brisket turn out on the new barbecue?", "Cooked it for twelve hours low and slow, neighbours came over and finished it all.", "Smoking a brisket for twelve hours on a new barbecue."),
    ("Hi Hannah, how is your daughter adjusting to starting kindergarten?", "Cried the first two mornings, but now she refuses to come home at pickup!", "Daughter starting kindergarten."),
    ("Morning Robert! Did the mechanics finally figure out that rattle in your pickup truck?", "Turned out to be a loose heat shield on the exhaust, only cost fifty bucks!", "Fixing a loose exhaust heat shield on a pickup truck."),
    ("Hi Maya, I saw the aurora borealis photos you posted from Boulevard Lake—stunning!", "Wasn't the northern lights display incredible? Stayed out until two in the morning taking pictures.", "Photographing the northern lights at Boulevard Lake."),
    ("Hey Jordan, how did the marathon training run go on Saturday?", "Wind off the lake was fierce, but I managed twenty kilometres without stopping.", "Twenty-kilometre training run in strong lake wind."),
    ("Morning Evelyn! Did your knitting circle finish the charity blankets for the shelter?", "Thirty-two blankets delivered yesterday! My hands need a week off.", "Knitting thirty-two charity blankets for a local shelter."),
    ("Hi Carlos, how is the new espresso machine treating you at home?", "Dangerously good, Doc—I'm practically running a café out of my kitchen now.", "Making coffee on a new home espresso machine."),
    ("Afternoon Rachel! Did you guys end up camping at Quetico Provincial Park in August?", "Four days of portaging canoes! Beautiful scenery, though the blackflies were relentless.", "Portaging canoes at Quetico Provincial Park."),
    ("Morning Victor! How did your chess tournament at the community centre go?", "Won three matches and drew the final one, came in second place overall!", "Finishing second place in a community centre chess tournament."),
    ("Hi Samantha, did you find a contractor to replace the roof shingles before winter?", "Finally got a crew coming Tuesday before the first heavy snowfall hits.", "Hiring a roofing crew before winter snowfall."),
    ("Hey Liam, how was the acoustic concert at the auditorium last Friday?", "Sound acoustics were phenomenal, they played a two-hour encore set.", "Attending an acoustic concert at the auditorium."),
    ("Morning Beatrice! How did your sourdough starter hold up while you were away?", "My neighbour fed it every day, baked two loaves of rye bread yesterday!", "Neighbour feeding a sourdough starter to bake rye bread."),
    ("Hi Owen, how is the high school robotics team looking for regionals?", "The kids built a pneumatic arm that stacks crates in ten seconds flat!", "Coaching a high school robotics team building a pneumatic arm."),
    ("Afternoon Diana! Did you manage to book flights for Christmas in Vancouver?", "Booked them last night, prices were sky-high but worth it to see the grandkids.", "Booking Christmas flights to Vancouver to see grandchildren."),
    ("Morning Felix! How is the ice thickness looking out on Lac des Mille Lacs?", "Still too thin for the ice hut, waiting another two weeks of sub-zero temps.", "Checking ice thickness on Lac des Mille Lacs for ice fishing."),
    ("Hi Tara, how did the pottery glazing class go at the art gallery?", "Made three ceramic mugs—one came out lopsided, so we call it abstract art!", "Making ceramic mugs at an art gallery pottery class."),
    ("Hey Noah, did the power outage during Wednesday's windstorm hit your street?", "Out for nine hours! We had to cook soup on the camping stove on the porch.", "Nine-hour power outage during a windstorm and cooking on a camp stove."),
    ("Morning Clara! How is your Persian cat recovering from the vet dental cleaning?", "Lost two teeth and is very grumpy about eating wet food, but doing fine!", "Persian cat recovering from a veterinary dental cleaning."),
    ("Hi Derek, how was the cross-country ski trail at Lappe Nordic this morning?", "Freshly groomed tracks, glided through the ten-kilometre loop in under an hour.", "Cross-country skiing at Lappe Nordic trails."),
    ("Afternoon Fiona! Did your daughter's piano recital go smoothly on Sunday?", "Not a single missed note on the Chopin piece, we went for gelato after.", "Daughter playing Chopin at a Sunday piano recital."),
    ("Morning George! How is retirement treating you after thirty years at the paper mill?", "Still waking up at five in the morning out of habit, but loving the quiet mornings!", "Retiring after thirty years working at the paper mill."),
    ("Hi Holly, did you manage to snag tickets for the symphony orchestra next month?", "Front row balcony! Really looking forward to hearing the cello soloist.", "Buying balcony tickets for the symphony orchestra."),
    ("Hey Ian, how did the deer hunting camp go up north last week?", "Didn't see a single buck all week, just played cards in the cabin and ate chili!", "Playing cards at deer hunting camp without seeing any deer."),
    ("Morning Julia! How is the new bookkeeping software working out at the bakery?", "Took a weekend to migrate the spreadsheets, but invoicing takes half the time now.", "Migrating bakery spreadsheets to new bookkeeping software."),
    ("Hi Kyle, did your snowblower start up okay after sitting in the shed all summer?", "Had to clean out the carburetor and put fresh gas in, runs like a champ now.", "Cleaning a snowblower carburetor before winter."),
    ("Afternoon Laura! How was the book club discussion on that historical novel?", "Half the group hated the ending, so we debated over tea for three hours!", "Debating a historical novel at a three-hour book club meeting."),
    ("Morning Malcolm! Did you finish restoring that vintage wooden canoe in your garage?", "Put the final coat of marine varnish on the cedar strips yesterday!", "Applying marine varnish to a vintage cedar-strip canoe."),
    ("Hi Nina, how are the rehearsals going for the community theatre play?", "Opening night is in ten days and our lead actor still forgets his Act Two cues!", "Rehearsing for a community theatre play opening in ten days."),
    ("Hey Oscar, how did the curling bonspiel in Kenora go over the weekend?", "Swept so hard my shoulders are still sore, made it to the semi-finals!", "Making the semi-finals at a curling bonspiel in Kenora."),
    ("Morning Paula! Did your son pass his G2 road test at the DriveTest centre?", "Passed on the first try, even parallel parked between two snowbanks!", "Son passing his G2 driving test and parallel parking."),
    ("Hi Quentin, how was the Persian New Year dinner at your cousin's place?", "So much herb rice and saffron chicken I could barely move for two days!", "Attending a family Persian New Year dinner."),
    ("Afternoon Rosa! Did the city crew finally clear the fallen birch tree off your boulevard?", "Came with the woodchipper at seven this morning, driveway is finally clear.", "City crew clearing a fallen birch tree with a woodchipper."),
    ("Morning Sam! How is your training going for the volunteer firefighter certification?", "Hose drills in full bunker gear yesterday—definitely a serious workout!", "Practicing hose drills in bunker gear for volunteer firefighting."),
    ("Hi Tessa, how did the science fair volcano project turn out with the twins?", "Baking soda and red vinegar erupted all over the kitchen ceiling, they loved it!", "Building a baking soda volcano with twins for a science fair."),
    ("Hey Victor, did you ever find your lost set of house keys after snowblowing?", "Found them frozen inside the snowbank right beside the mailbox this morning!", "Finding lost house keys frozen in a snowbank."),
    ("Morning Wendy! How is the watercolour landscape painting class progressing?", "Learning how to paint birch trees in winter shadows—harder than it looks!", "Taking a winter watercolour landscape painting class."),
    ("Hi Xavier, how was the amethyst mine tour you took your nephew on?", "He dug up a four-pound purple crystal cluster and carries it everywhere now!", "Visiting a local amethyst mine with a nephew."),
    ("Afternoon Yasmine! Did the new triple-pane windows make a difference with the draft?", "Living room stays three degrees warmer and we can't hear street traffic at all!", "Installing new triple-pane windows in the living room.")
]

# 2. Define 50 distinct Clinical Case Templates across 10 Specialties
#    Each tuple provides:
#    (Specialty, ICD10, Patient_Complaint_Lines, Doc_Question_Lines, Patient_Followup_Lines,
#     Doc_Exam_And_Plan_Lines, Patient_Alternative_Q, Doc_PICO_Justification_Lines,
#     Linguistic_Shifts_Dict, Omitted_Fields_List, PICO_Dict, SOAP_Sections_Dict)

CLINICAL_CASES_SPEC = [
    # --- CARDIOLOGY (EXP_001 to EXP_005) ---
    ("Cardiology", "R00.2",
     "Anyway, I'm here because for the past two weeks, my heart is fluttering like a bird in my chest whenever I climb the stairs.",
     "I'm sorry to hear that. Does it happen at rest, or do you ever feel dizzy or get winded?",
     "Sometimes when I'm watching TV at night it races too. And yeah, I get winded way faster than usual. No chest pain though.",
     "Blood pressure is 138/84 mmHg, pulse is 92 bpm and slightly irregular. Lungs are clear. I want to order a 48-hour Holter monitor and consider Metoprolol 25mg daily.",
     "Do I have to take pills right away? Can I just cut out my morning coffee instead?",
     "Cutting out caffeine is a good conservative alternative alongside monitoring, so let's hold the Metoprolol for 48 hours while we run the Holter monitor first, then decide in one week.",
     {"heart is fluttering": "intermittent palpitations", "it races": "tachycardia", "get winded": "exertional dyspnea"},
     ["Allergies", "Family History", "Temperature", "Respiratory Rate"],
     {"Patient_Diagnosis": "Adult patient presenting with intermittent palpitations, nocturnal tachycardia, and exertional dyspnea; suspected cardiac arrhythmia.",
      "Intervention": "48-hour ambulatory Holter ECG monitor combined with caffeine cessation (deferring Metoprolol 25mg daily pending tracing).",
      "Comparison_Alternative": "Immediate beta-blocker initiation (Metoprolol 25mg PO daily) versus diagnostic Holter monitoring with conservative caffeine elimination.",
      "Outcome": "Capture underlying arrhythmic events on 48-hour ECG and evaluate palpitation reduction at 1-week follow-up."},
     {"CC": "Intermittent palpitations and exertional dyspnea for 2 weeks.",
      "HPI": "Patient reports intermittent palpitations ('fluttering') provoked by climbing stairs and nocturnal episodes of tachycardia at rest, accompanied by exertional dyspnea. Denies chest pain or syncope.",
      "Vitals": "BP 138/84 mmHg, HR 92 bpm (irregular), Temp: Not Documented, RR: Not Documented.",
      "Exam": "Lungs clear to auscultation bilaterally; irregular cardiac rhythm.",
      "Assessment": "Suspected cardiac arrhythmia presenting with intermittent palpitations and exertional dyspnea."}
    ),
    ("Cardiology", "I10",
     "So the reason I came in is my home cuff keeps showing sky-high blood pressure numbers around 165 over 100, and I have a dull throbbing in the back of my neck every morning.",
     "Are you having any blurry vision, chest tightness, or swelling in your ankles?",
     "No blurry vision or chest tightness, and my ankles look totally normal.",
     "In clinic today your blood pressure is 162/98 mmHg averaged over three readings, heart rate 76 bpm. Heart sounds are normal. Because you are consistently in stage 2 hypertension, I recommend starting Lisinopril 10mg once daily.",
     "Can I just do a low-salt diet and walk more for a few months instead of starting blood pressure pills?",
     "While sodium restriction is essential, at 162/98 mmHg lifestyle changes alone typically lower pressure by only 5 to 8 points, whereas adding an ACE inhibitor like Lisinopril reliably prevents cardiovascular strain. Let's combine both and recheck in 4 weeks.",
     {"sky-high blood pressure": "stage 2 systemic hypertension", "dull throbbing in the back of my neck": "occipital cephalalgia", "swelling in your ankles": "bilateral pedal edema", "blurry vision": "visual disturbance"},
     ["Allergies", "Smoking History", "Temperature", "Respiratory Rate"],
     {"Patient_Diagnosis": "Uncontrolled Stage 2 Essential Hypertension (BP 162/98 mmHg) with morning occipital cephalalgia.",
      "Intervention": "Lisinopril 10mg PO daily combined with dietary sodium restriction and aerobic exercise.",
      "Comparison_Alternative": "Lifestyle modification monotherapy (low-sodium diet and walking) versus combined ACE inhibitor pharmacotherapy (Lisinopril 10mg).",
      "Outcome": "Achieve target blood pressure <130/80 mmHg and resolve morning occipital headaches at 4-week follow-up."},
     {"CC": "Elevated home blood pressure readings and morning occipital cephalalgia.",
      "HPI": "Patient presents with persistently elevated home BP readings (~165/100 mmHg) and morning occipital headaches. Denies visual disturbances, chest pain, or bilateral pedal edema.",
      "Vitals": "BP 162/98 mmHg (3-reading average), HR 76 bpm, Temp: Not Documented, RR: Not Documented.",
      "Exam": "Normal S1/S2 heart sounds without murmurs; no peripheral edema.",
      "Assessment": "Stage 2 Essential Hypertension."}
    ),
    ("Cardiology", "I20.9",
     "Listen Doc, for the past three weeks, whenever I shovel snow in the driveway, I get this heavy elephant sitting on my chest and an ache going down my left arm.",
     "How long does that heavy feeling last when you stop shoveling, and does it make you break out in a cold sweat?",
     "Once I come inside and sit down for five minutes it goes away completely. A little sweaty, yeah.",
     "Resting blood pressure is 144/88 mmHg, pulse 80 bpm, resting ECG is normal sinus rhythm. This pattern is classic for stable exertional angina. I'm prescribing sublingual Nitroglycerin 0.4mg spray for acute episodes, daily Aspirin 81mg, and ordering an exercise stress echocardiogram.",
     "Why do I need a stress test if my resting ECG right now looks completely normal?",
     "A resting ECG only checks your heart at idle and misses blockages that only starve the heart muscle during heavy work, whereas an exercise stress echocardiogram visualizes blood flow under load. Avoid snow shoveling until we complete the test next week.",
     {"heavy elephant sitting on my chest": "substernal chest pressure / angina pectoris", "ache going down my left arm": "left brachial radiation", "break out in a cold sweat": "diaphoresis"},
     ["Medication Allergies", "Family History", "Temperature", "Oxygen Saturation"],
     {"Patient_Diagnosis": "Suspected coronary artery disease presenting with classic stable exertional angina pectoris.",
      "Intervention": "Exercise stress echocardiogram, Aspirin 81mg PO daily, and PRN sublingual Nitroglycerin 0.4mg spray.",
      "Comparison_Alternative": "Exercise stress echocardiogram + antiplatelet/nitrate therapy versus relying solely on a normal resting 12-lead ECG.",
      "Outcome": "Evaluate inducible myocardial ischemia on stress imaging and prevent acute coronary events."},
     {"CC": "Exertional substernal chest pressure radiating to the left arm for 3 weeks.",
      "HPI": "Patient reports substernal chest pressure ('elephant on chest') with left brachial radiation and mild diaphoresis triggered by snow shoveling, resolving within 5 minutes of rest.",
      "Vitals": "BP 144/88 mmHg, HR 80 bpm, Temp: Not Documented, SpO2: Not Documented.",
      "Exam": "Resting 12-lead ECG shows normal sinus rhythm without acute ST-T wave changes.",
      "Assessment": "Stable exertional angina pectoris; suspected obstructive coronary artery disease."}
    ),
    ("Cardiology", "I50.9",
     "I'm really struggling lately, Doc. I have to prop myself up on three pillows at night just to breathe, and my shoes don't fit because my feet and calves are puffed up like balloons.",
     "Do you ever wake up gasping for air in the middle of the night or cough up pink foamy spit?",
     "Yeah, twice this week I bolted upright at 2 AM gasping for air by the window. No pink spit though.",
     "Weight is up 4 kg from last month to 94 kg, BP is 148/90 mmHg, oxygen saturation is 93% on room air. I hear crackles at both lung bases and see 2+ pitting edema up to your mid-shins. You have mild fluid overload from congestive heart failure. We need to start Furosemide 40mg daily and check a BNP blood test and echocardiogram.",
     "Can I just drink way less water and wear compression socks instead of taking a water pill?",
     "Compression stockings only push fluid around without removing it from your lungs, whereas a loop diuretic like Furosemide actively clears excess fluid through your kidneys. Weigh yourself daily and return in 5 days for electrolyte and kidney checks.",
     {"prop myself up on three pillows to breathe": "orthopnea (3-pillow)", "feet and calves are puffed up like balloons": "bilateral lower extremity pitting edema", "wake up gasping for air in the middle of the night": "paroxysmal nocturnal dyspnea (PND)", "pink foamy spit": "pink frothy sputum"},
     ["Allergies", "Smoking History", "Temperature", "Heart Rate"],
     {"Patient_Diagnosis": "Decompensated Congestive Heart Failure (CHF) presenting with 3-pillow orthopnea, PND, and bilateral pedal edema.",
      "Intervention": "Furosemide 40mg PO daily, serum BNP, transthoracic echocardiogram, and daily weight monitoring.",
      "Comparison_Alternative": "Loop diuretic pharmacotherapy (Furosemide 40mg daily) versus non-pharmacological fluid restriction and compression stockings alone.",
      "Outcome": "Diuresis of excess pulmonary and peripheral fluid, resolution of orthopnea/PND, and target weight reduction at 5-day follow-up."},
     {"CC": "Orthopnea, paroxysmal nocturnal dyspnea, and bilateral lower extremity edema.",
      "HPI": "Patient reports 3-pillow orthopnea, two episodes of paroxysmal nocturnal dyspnea this week, 4 kg weight gain, and bilateral lower extremity swelling. Denies hemoptysis or pink frothy sputum.",
      "Vitals": "BP 148/90 mmHg, SpO2 93% (room air), Weight 94 kg, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Bibasilar pulmonary crackles (rales); 2+ bilateral pitting edema to mid-tibia.",
      "Assessment": "Acute-on-chronic Congestive Heart Failure exacerbation with volume overload."}
    ),
    ("Cardiology", "R55",
     "Doc, I had a scary moment yesterday—when I stood up fast from the couch, the whole room went black and I passed out cold on the carpet for ten seconds.",
     "Did anyone see you shaking on the floor, or did you bite your tongue or lose bladder control?",
     "My wife was right there—she said I didn't shake at all and woke up right away feeling fine, just a bit woozy when standing.",
     "Sitting blood pressure is 126/78 mmHg with heart rate 68 bpm, but when you stand up for three minutes it drops to 98/62 mmHg with heart rate 88 bpm. Heart sounds and neuro exam are normal. This is orthostatic syncope, likely exacerbated by the Tamsulosin you take at night. Let's switch your Tamsulosin timing, increase hydration and electrolytes, and review your med list.",
     "Should we order an urgent brain MRI Scan to make sure I didn't have a mini-stroke?",
     "Brain MRI is unhelpful for brief fainting triggered by standing with a confirmed 28-point blood pressure drop, whereas correcting orthostatic hypotension via medication adjustment and hydration directly prevents recurrence. Follow up in 2 weeks.",
     {"the whole room went black": "presyncopal visual dimming / tunnel vision", "passed out cold": "syncope (transient loss of consciousness)", "shaking on the floor": "tonic-clonic seizure activity", "woozy when standing": "orthostatic lightheadedness"},
     ["Medication Allergies", "Family History", "Temperature", "Respiratory Rate"],
     {"Patient_Diagnosis": "Orthostatic syncope secondary to alpha-blocker-induced orthostatic hypotension.",
      "Intervention": "Medication schedule modification, increased fluid/electrolyte intake, and counter-pressure rising maneuvers.",
      "Comparison_Alternative": "Conservative hemodynamic management and medication adjustment versus routine neuroimaging (Brain MRI) for uncomplicated orthostatic syncope.",
      "Outcome": "Elimination of orthostatic blood pressure drops and prevention of recurrent syncopal episodes at 2-week follow-up."},
     {"CC": "Single episode of syncope upon standing yesterday.",
      "HPI": "Patient experienced presyncopal visual dimming followed by brief (~10-second) syncope upon standing rapidly from seated position. Witnessed by spouse; negative for tonic-clonic activity, tongue biting, or incontinence.",
      "Vitals": "Sitting BP 126/78 mmHg (HR 68 bpm); Standing BP 98/62 mmHg (HR 88 bpm); Temp: Not Documented, RR: Not Documented.",
      "Exam": "Positive orthostatic vital sign drop (>20 mmHg systolic); cardiovascular and neurological exams intact.",
      "Assessment": "Orthostatic syncope secondary to orthostatic hypotension."}
    ),

    # --- GASTROENTEROLOGY (EXP_006 to EXP_010) ---
    ("Gastroenterology", "K21.9",
     "Look Doc, for a month now I've had this awful burning fire in my throat every time I lie down after dinner, and I keep burping up sour acid.",
     "Any trouble swallowing solid food, throwing up blood, or black tarry stools?",
     "No, none of that. Just feels like food sits like a brick in my stomach.",
     "Your abdomen is soft and non-tender today, weight is stable at 68 kg. This is classic gastroesophageal reflux. I'm prescribing Omeprazole 20mg once daily before breakfast for 4 weeks.",
     "I've been chewing Tums at night—can I just keep doing that instead of a daily prescription?",
     "Over-the-counter calcium carbonate antacids only neutralize acid briefly, whereas a proton pump inhibitor like Omeprazole suppresses acid production long enough for esophageal tissue to heal. Let's do the 4-week course and elevate your bedhead.",
     {"burning fire in my throat": "pyrosis (heartburn)", "burping up sour acid": "acid regurgitation", "trouble swallowing": "dysphagia", "throwing up blood": "hematemesis", "black tarry stools": "melena", "food sits like a brick": "postprandial fullness / dyspepsia"},
     ["Blood Pressure", "Heart Rate", "Allergies", "Past Surgical History"],
     {"Patient_Diagnosis": "Gastroesophageal Reflux Disease (GERD) presenting with nocturnal pyrosis and acid regurgitation.",
      "Intervention": "Omeprazole 20mg PO daily before breakfast for 4 weeks plus nocturnal head-of-bed elevation.",
      "Comparison_Alternative": "Daily Proton Pump Inhibitor therapy (Omeprazole 20mg) versus PRN over-the-counter antacids (calcium carbonate / Tums).",
      "Outcome": "Mucosal healing and complete resolution of nocturnal pyrosis and regurgitation at 4 weeks."},
     {"CC": "Nocturnal pyrosis and acid regurgitation for 1 month.",
      "HPI": "Patient reports postprandial dyspepsia, nocturnal pyrosis, and acid regurgitation when recumbent. Denies dysphagia, hematemesis, or melena.",
      "Vitals": "Weight 68 kg, BP: Not Documented, HR: Not Documented.",
      "Exam": "Abdomen soft, non-tender, non-distended.",
      "Assessment": "Uncomplicated Gastroesophageal Reflux Disease (GERD)."}
    ),
    ("Gastroenterology", "K80.20",
     "For the last two weeks, every time I eat greasy food like fish and chips, I get this stabbing knife pain under my right ribs that shoots straight to my right shoulder blade.",
     "Does the pain come with fever, yellowing of your eyes, or throwing up?",
     "Felt pretty queasy and threw up once after a burger on Tuesday, but no fever and my eyes aren't yellow.",
     "Temperature is 36.8°C, BP 128/82 mmHg. You have marked tenderness in the right upper quadrant when taking a deep breath, which is a positive Murphy's sign. I'm ordering an abdominal ultrasound and liver blood panel to check for gallstones, and advising a strict low-fat diet while we arrange a surgical referral.",
     "If the ultrasound shows gallstones, can we just dissolve them with medicine or sound waves instead of taking my gallbladder out?",
     "Oral bile acid pills take up to two years and gallstones come back in half of patients, whereas laparoscopic cholecystectomy permanently cures symptomatic biliary colic. Let's get the ultrasound done this week first.",
     {"stabbing knife pain under my right ribs": "right upper quadrant (RUQ) biliary colic", "shoots straight to my right shoulder blade": "radiation to the right scapula (Boas sign)", "yellowing of your eyes": "scleral icterus / jaundice", "felt pretty queasy and threw up": "nausea and emesis"},
     ["Heart Rate", "Respiratory Rate", "Allergies", "Family History"],
     {"Patient_Diagnosis": "Symptomatic cholelithiasis (biliary colic) with postprandial RUQ pain radiating to the right scapula.",
      "Intervention": "Right upper quadrant abdominal ultrasound, hepatic function panel, low-fat diet, and general surgery referral for laparoscopic cholecystectomy evaluation.",
      "Comparison_Alternative": "Definitive laparoscopic cholecystectomy versus medical dissolution therapy (ursodeoxycholic acid) or lithotripsy.",
      "Outcome": "Confirm cholelithiasis on imaging and prevent acute cholecystitis or choledocholithiasis."},
     {"CC": "Postprandial right upper quadrant pain radiating to the right scapula for 2 weeks.",
      "HPI": "Patient reports episodic severe RUQ pain triggered by fatty meals with radiation to the right scapula, associated with nausea and single episode of emesis. Denies fever or scleral icterus.",
      "Vitals": "Temp 36.8°C, BP 128/82 mmHg, HR: Not Documented, RR: Not Documented.",
      "Exam": "RUQ abdominal tenderness with inspiratory arrest on palpation (positive Murphy's sign); anicteric sclerae.",
      "Assessment": "Biliary colic / suspected symptomatic cholelithiasis."}
    ),
    ("Gastroenterology", "K58.0",
     "My stomach has been driving me crazy for four months, Doc. My belly blows up like a beach ball every afternoon with cramping, and I'm running to the bathroom with loose watery stools three times a day.",
     "Does the cramping feel better right after you have a bowel movement, and have you ever seen red blood in the toilet or lost weight?",
     "Yeah, going to the bathroom relieves the cramps right away! No blood at all, and my weight hasn't changed.",
     "Abdominal exam shows mild diffuse tenderness without guarding or masses. Celiac screen and stool inflammatory markers (fecal calprotectin) from last week are completely normal. This meets criteria for diarrhea-predominant Irritable Bowel Syndrome (IBS-D). I recommend starting a structured Low-FODMAP diet with soluble psyllium fiber and peppermint oil capsules.",
     "Shouldn't I get a full colonoscopy right now just to be 100% sure?",
     "In patients under 45 with negative fecal calprotectin, normal celiac serology, and zero alarm symptoms like bleeding or weight loss, invasive colonoscopy has extremely low yield compared to targeted Low-FODMAP dietary management. Let's review your symptom diary in 6 weeks.",
     {"belly blows up like a beach ball": "abdominal distension and bloating", "running to the bathroom with loose watery stools": "chronic diarrhea with fecal urgency", "red blood in the toilet": "hematochezia"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Diarrhea-Predominant Irritable Bowel Syndrome (IBS-D) per Rome IV criteria.",
      "Intervention": "Structured Low-FODMAP dietary protocol, soluble psyllium fiber supplementation, and enteric-coated peppermint oil.",
      "Comparison_Alternative": "Non-invasive dietary/medical IBS-D management (guided by negative fecal calprotectin) versus immediate diagnostic colonoscopy in the absence of alarm features.",
      "Outcome": "Reduction in abdominal bloating, cramping, and stool frequency at 6-week follow-up."},
     {"CC": "Chronic abdominal bloating, cramping relieved by defecation, and loose stools for 4 months.",
      "HPI": "Patient reports daily afternoon abdominal distension, crampy abdominal pain relieved by defecation, and 3 loose watery stools daily. Denies hematochezia, nocturnal diarrhea, or weight loss.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Abdomen soft with mild diffuse tenderness; no guarding, rebound, or palpable masses. Normal celiac panel and fecal calprotectin.",
      "Assessment": "Irritable Bowel Syndrome, diarrhea-predominant (IBS-D)."}
    ),
    ("Gastroenterology", "K25.9",
     "Doc, for three weeks I've had this gnawing hunger-like pain right in the pit of my stomach that wakes me up at 3 AM. Oddly enough, eating a cracker makes it feel better for an hour.",
     "Have you been taking ibuprofen or naproxen for joint pain, and what colour are your bowel movements?",
     "I've been popping two naproxen every day for my knee for two months. Stools look normal brown, not black.",
     "Blood pressure is 122/76 mmHg, pulse 74 bpm. Epigastric palpation reproduces mild tenderness without peritoneal signs. Suspect NSAID-induced peptic ulcer disease. We must stop the naproxen immediately, run a stool H. pylori antigen test, and start Pantoprazole 40mg daily for 8 weeks.",
     "Can I just switch from Naproxen to Aspirin or take the Naproxen with milk instead?",
     "Taking NSAIDs with milk or switching to Aspirin still blocks protective stomach prostaglandins and risks ulcer bleeding, whereas discontinuing NSAIDs and using Pantoprazole 40mg allows the gastric mucosa to heal. Use acetaminophen if needed for knee pain.",
     {"gnawing hunger-like pain right in the pit of my stomach": "epigastric pain", "wakes me up at 3 AM": "nocturnal epigastric pain", "black stools": "melena"},
     ["Temperature", "Respiratory Rate", "Allergies", "Family History"],
     {"Patient_Diagnosis": "Suspected NSAID-induced Peptic Ulcer Disease (PUD) presenting with nocturnal epigastric pain.",
      "Intervention": "Immediate cessation of Naproxen, Pantoprazole 40mg PO daily for 8 weeks, and stool Helicobacter pylori antigen testing.",
      "Comparison_Alternative": "Complete NSAID cessation + Proton Pump Inhibitor therapy versus continuing NSAIDs with milk/food or switching to Aspirin.",
      "Outcome": "Complete peptic ulcer mucosal healing, H. pylori eradication if positive, and resolution of epigastric pain at 8 weeks."},
     {"CC": "Epigastric pain waking patient at night for 3 weeks.",
      "HPI": "Patient reports gnawing epigastric pain relieved temporarily by food and waking them at 3 AM, in the setting of daily Naproxen use for 2 months. Denies melena or hematemesis.",
      "Vitals": "BP 122/76 mmHg, HR 74 bpm, Temp: Not Documented, RR: Not Documented.",
      "Exam": "Localized epigastric tenderness on palpation without rebound or guarding.",
      "Assessment": "Suspected NSAID-induced Peptic Ulcer Disease."}
    ),
    ("Gastroenterology", "K59.00",
     "I'm really uncomfortable, Doc. I haven't had a proper bowel movement in five days, and when I do go, it's just hard little rabbit pellets and I have to strain like crazy.",
     "Any sharp belly pain, throwing up, or thin pencil-like stools?",
     "No throwing up and no pencil stools, just feeling super backed up and sluggish.",
     "Abdomen is mildly distended with palpable stool in the left lower quadrant, normal bowel sounds. Thyroid lab test from last month was normal. I recommend starting daily Polyethylene Glycol (PEG 3350 / Lax-A-Day) 17g in water along with increasing fluid and dietary fiber.",
     "My friend told me to drink senna laxative tea every night—isn't that more natural and faster?",
     "Chronic stimulant laxatives like senna irritate bowel nerves and cause cramping, whereas osmotic laxatives like Polyethylene Glycol gently draw water into the stool safely for long-term regularity. Follow up in 4 weeks.",
     {"hard little rabbit pellets": "scybalous stools (Bristol Stool Scale Type 1)", "strain like crazy": "excessive defecatory straining (dyschezia)", "feeling super backed up": "functional constipation"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Chronic functional constipation.",
      "Intervention": "Polyethylene Glycol 3350 (17g PO daily in water) combined with increased dietary fiber and oral hydration.",
      "Comparison_Alternative": "Osmotic laxative therapy (PEG 3350) versus chronic daily stimulant laxative use (senna tea).",
      "Outcome": "Restore regular soft bowel movements (Bristol Type 3-4) without straining at 4-week follow-up."},
     {"CC": "Decreased bowel movement frequency (every 5 days) and hard stools with straining.",
      "HPI": "Patient reports passage of Bristol Type 1 scybalous stools with excessive straining and abdominal fullness. Denies emesis, hematochezia, or caliber change.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Normoactive bowel sounds; mild LLQ fullness consistent with fecal loading; no peritoneal signs.",
      "Assessment": "Functional constipation."}
    ),

    # --- PULMONOLOGY (EXP_011 to EXP_015) ---
    ("Pulmonology", "J45.901",
     "Actually, ever since three weeks ago, my chest feels super tight and I'm whistling when I breathe out, especially at night.",
     "Have you been coughing up any phlegm or running a fever?",
     "Dry hacking cough all night, no fever at all. I'm totally exhausted.",
     "Oxygen saturation is 96% on room air, respiratory rate is 20 breaths/min. I hear diffuse expiratory wheezes across both lung fields. Allergic asthma is very likely. I want to start you on a low-dose inhaled budesonide-formoterol (Symbicort) inhaler as both maintenance and reliever.",
     "My brother uses just a blue puffer—salbutamol—whenever he gets tight. Why not just give me that?",
     "Salbutamol alone only opens airways temporarily without treating underlying airway inflammation, whereas combination budesonide-formoterol treats inflammation and cuts severe asthma flare-ups significantly. Let's review in 4 weeks.",
     {"chest feels super tight": "chest tightness / bronchospasm", "whistling when I breathe out": "expiratory wheezing", "coughing up any phlegm": "productive sputum", "dry hacking cough": "non-productive nocturnal cough"},
     ["Blood Pressure", "Heart Rate", "Medication Allergies", "Smoking History"],
     {"Patient_Diagnosis": "Mild-to-moderate allergic asthma exacerbation.",
      "Intervention": "Inhaled budesonide-formoterol (Symbicort) maintenance and reliever therapy (MART).",
      "Comparison_Alternative": "Anti-inflammatory ICS-formoterol combination inhaler versus PRN short-acting beta-agonist (Salbutamol) monotherapy.",
      "Outcome": "Resolution of nocturnal cough/wheezing and reduction in airway inflammation at 4-week follow-up."},
     {"CC": "Chest tightness, expiratory wheezing, and nocturnal dry cough for 3 weeks.",
      "HPI": "Patient presents with nocturnal chest tightness, expiratory wheezing, and non-productive hacking cough. Denies fever or sputum production.",
      "Vitals": "SpO2 96% (room air), RR 20 breaths/min, BP: Not Documented, HR: Not Documented.",
      "Exam": "Diffuse bilateral expiratory wheezes on lung auscultation.",
      "Assessment": "Allergic asthma exacerbation."}
    ),
    ("Pulmonology", "J44.1",
     "Doc, my smoker's cough has gotten way worse over the last four days—I'm coughing up thick yellowish-green gunk and I can barely walk from the kitchen to the bathroom without gasping.",
     "How many packs a day have you smoked, and do you have a fever or chest pain?",
     "Smoked a pack a day for thirty years. No chest pain, felt a tiny bit warm last night.",
     "Temperature is 37.4°C, oxygen saturation is 90% on room air, respiratory rate 24. Decreased breath sounds with coarse rhonchi bilaterally. This is an acute exacerbation of COPD. I'm prescribing Prednisone 40mg daily for 5 days, Amoxicillin-Clavulanate for 5 days, and stepping up your inhaled bronchodilators.",
     "I hate how jittery Prednisone makes me feel—can I just take the antibiotic and skip the steroid pills?",
     "While antibiotics clear bacterial infection, oral Prednisone is critical for rapidly reducing severe bronchial swelling and improving oxygen levels during an acute COPD flare. A short 5-day burst minimizes side effects. Re-evaluate in 48 hours.",
     {"coughing up thick yellowish-green gunk": "increased purulent sputum production", "barely walk from the kitchen to the bathroom without gasping": "severe resting/minimal-exertion dyspnea", "smoker's cough": "chronic bronchitis"},
     ["Blood Pressure", "Heart Rate", "Medication Allergies", "Family History"],
     {"Patient_Diagnosis": "Acute Infectious Exacerbation of Chronic Obstructive Pulmonary Disease (AECOPD).",
      "Intervention": "Prednisone 40mg PO daily x 5 days, Amoxicillin-Clavulanate PO x 5 days, and intensified inhaled bronchodilator therapy.",
      "Comparison_Alternative": "Combined systemic corticosteroid + antibiotic + bronchodilator therapy versus antibiotic monotherapy without systemic steroids.",
      "Outcome": "Improvement in SpO2 (>92%), reduction in sputum purulence, and stabilization of dyspnea within 48–72 hours."},
     {"CC": "Worsening dyspnea and purulent sputum production for 4 days.",
      "HPI": "Patient with 30-pack-year smoking history presents with acute worsening of dyspnea at minimal exertion and increased yellowish-green purulent sputum. Denies chest pain.",
      "Vitals": "Temp 37.4°C, SpO2 90% (room air), RR 24 breaths/min, BP: Not Documented, HR: Not Documented.",
      "Exam": "Diminished vesicular breath sounds with bilateral coarse rhonchi.",
      "Assessment": "Acute exacerbation of COPD (AECOPD)."}
    ),
    ("Pulmonology", "J18.9",
     "I've been sick as a dog for five days, Doc. Shaking chills, high fever, and a deep rattle in my right chest coughing up rust-coloured mucus.",
     "Does it hurt sharply in your chest when you take a deep breath in?",
     "Yes! Every time I breathe in deep it stabs right in my lower right ribcage.",
     "Temperature is 38.9°C, heart rate 104 bpm, respiratory rate 22, SpO2 94% on room air. Right lower lung base shows crackles and dullness on percussion. Chest X-ray confirms a right lower lobe consolidation. You have community-acquired pneumonia. I'm starting Amoxicillin 1g three times daily plus Azithromycin for 5 days.",
     "Do I need to be admitted to the hospital for IV antibiotics right now?",
     "Because your oxygen is 94%, blood pressure is stable, and your CURB-65 severity score is low, oral high-dose Amoxicillin plus Azithromycin at home is equally effective as hospital IV therapy and avoids hospital-acquired infections. Follow up in 48 hours.",
     {"shaking chills": "rigors", "rust-coloured mucus": "rust-colored hemoptysis/sputum", "stabs right in my lower right ribcage when I breathe in": "right-sided pleuritic chest pain"},
     ["Blood Pressure", "Allergies", "Smoking History", "Past Medical History"],
     {"Patient_Diagnosis": "Right Lower Lobe Community-Acquired Pneumonia (CAP), low-risk CURB-65.",
      "Intervention": "Outpatient oral Amoxicillin 1g TID + Azithromycin x 5 days, hydration, and 48-hour clinical reassessment.",
      "Comparison_Alternative": "Outpatient oral combination antibiotic therapy (guided by low CURB-65 score) versus inpatient hospitalization for IV antibiotics.",
      "Outcome": "Defervescence within 48–72 hours and radiographic/clinical resolution of right lower lobe pneumonia."},
     {"CC": "Fever, rigors, productive cough with rust-colored sputum, and pleuritic chest pain for 5 days.",
      "HPI": "Patient reports 5-day history of high fever, rigors, productive cough with rust-colored sputum, and right-sided pleuritic chest pain.",
      "Vitals": "Temp 38.9°C, HR 104 bpm, RR 22 breaths/min, SpO2 94% (room air), BP: Not Documented.",
      "Exam": "Right lower lobe crackles and dullness to percussion; CXR confirms RLL consolidation.",
      "Assessment": "Right lower lobe Community-Acquired Pneumonia (CAP)."}
    ),
    ("Pulmonology", "G47.33",
     "My partner made me book this appointment because she says I snore like a chainsaw all night and literally stop breathing for ten seconds before choking awake.",
     "How do you feel during the daytime? Do you ever nod off while reading or watching TV, or wake up with a dry mouth?",
     "I wake up with a desert-dry mouth and a headache every morning, and I almost nodded off at my desk at 2 PM yesterday.",
     "BMI is 34 kg/m2, blood pressure is 142/88 mmHg, neck circumference is 44 cm, and Mallampati score is Class III. This strongly points to Obstructive Sleep Apnea (OSA). I am ordering an overnight polysomnography sleep study to titrate for CPAP therapy.",
     "Can I just buy one of those nasal strip stickers or a chin strap online instead of doing a sleep study and CPAP?",
     "Adhesive nasal strips only widen the nostrils and do nothing for posterior pharyngeal airway collapse, whereas CPAP provides positive air pressure that prevents nocturnal hypoxia and lowers your blood pressure. Let's complete the sleep study first.",
     {"snore like a chainsaw": "loud disruptive snoring", "stop breathing and choke awake": "witnessed nocturnal apneas and gasping arousals", "nod off at my desk": "excessive daytime somnolence", "desert-dry mouth": "nocturnal xerostomia"},
     ["Heart Rate", "Respiratory Rate", "Allergies", "Smoking History"],
     {"Patient_Diagnosis": "High clinical probability of Obstructive Sleep Apnea (OSA) with excessive daytime somnolence.",
      "Intervention": "Diagnostic overnight polysomnography (sleep study) followed by Continuous Positive Airway Pressure (CPAP) titration.",
      "Comparison_Alternative": "Polysomnography-guided CPAP therapy versus over-the-counter external nasal dilator strips or chin straps.",
      "Outcome": "Elimination of nocturnal apneic episodes (AHI <5/hr), normalization of daytime alertness, and improved blood pressure control."},
     {"CC": "Loud snoring, witnessed nocturnal apneas, and excessive daytime somnolence.",
      "HPI": "Spouse reports loud snoring and witnessed apneic pauses followed by choking arousals. Patient reports morning xerostomia, morning headaches, and daytime somnolence.",
      "Vitals": "BP 142/88 mmHg, BMI 34 kg/m2, HR: Not Documented, RR: Not Documented.",
      "Exam": "Neck circumference 44 cm; Mallampati Class III oropharynx.",
      "Assessment": "Suspected Obstructive Sleep Apnea (OSA)."}
    ),
    ("Pulmonology", "J20.9",
     "Doc, I had a regular head cold last week, and now I'm stuck with this barking seal cough for eight days that keeps me up. I bring up a tiny bit of clear phlegm.",
     "Do you have any fever, shortness of breath, or history of asthma?",
     "No fever at all, breathing feels fine, just my ribs ache from coughing so much.",
     "Temperature is 36.7°C, heart rate 72 bpm, oxygen saturation 99% on room air, and your lungs are 100% clear with zero wheezes or crackles. This is acute viral bronchitis. I recommend honey, dextromethorphan lozenges at night, and a cool-mist humidifier.",
     "Can you please just prescribe me a Z-pack antibiotic so I can knock this cough out before Monday?",
     "Over 90% of acute bronchitis is viral, so Azithromycin will not shorten your cough by a single day and only risks gut side effects and antibiotic resistance, whereas supportive care lets the airway lining heal naturally over 10 to 14 days.",
     {"barking seal cough": "paroxysmal harsh cough", "clear phlegm": "mucoid sputum", "ribs ache from coughing": "post-tussive chest wall soreness"},
     ["Blood Pressure", "Respiratory Rate", "Medication Allergies", "Family History"],
     {"Patient_Diagnosis": "Uncomplicated Acute Viral Bronchitis.",
      "Intervention": "Supportive symptomatic management (dextromethorphan antitussive lozenges, honey, hydration, humidification) and antibiotic stewardship counseling.",
      "Comparison_Alternative": "Supportive symptomatic therapy versus unnecessary macrolide antibiotic therapy (Azithromycin / Z-pack).",
      "Outcome": "Gradual resolution of post-viral bronchial hyperresponsiveness within 10–14 days without antibiotic-associated adverse events."},
     {"CC": "Persistent paroxysmal cough with clear mucoid sputum for 8 days following a URI.",
      "HPI": "Patient reports 8-day history of harsh cough with scant clear sputum and post-tussive chest wall soreness after a viral URI. Denies fever or dyspnea.",
      "Vitals": "Temp 36.7°C, HR 72 bpm, SpO2 99% (room air), BP: Not Documented, RR: Not Documented.",
      "Exam": "Lungs clear to auscultation bilaterally without wheezes, rales, or consolidation.",
      "Assessment": "Acute viral bronchitis."}
    ),

    # --- NEUROLOGY (EXP_016 to EXP_020) ---
    ("Neurology", "G43.109",
     "I've had three massive headaches this month where the right side of my head feels like it's pounding with a jackhammer. Right before it starts, I see sparkly zigzag lines for 20 minutes.",
     "Does light or noise bother you when the pounding starts? Any throwing up?",
     "Yes! I have to lock myself in a dark room with earplugs, and I feel super nauseous, though I haven't actually thrown up.",
     "Your neurological exam—cranial nerves II-XII, motor strength, and coordination—is completely normal today. You have classic migraine with visual aura. I recommend Sumatriptan 50mg taken at the very first sign of the attack.",
     "I've been taking two Advil, doesn't that do the same thing?",
     "Over-the-counter Ibuprofen helps mild tension headaches, but Triptans specifically target serotonin receptors to abort migraine cascades. Take Sumatriptan at onset and keep a headache diary for the next month.",
     {"pounding with a jackhammer on right side": "unilateral pulsatile cephalalgia", "sparkly zigzag lines": "visual aura (scintillating scotoma)", "light or noise bother you": "photophobia and phonophobia", "throwing up": "emesis"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Family History", "Allergies"],
     {"Patient_Diagnosis": "Migraine with visual aura (scintillating scotoma).",
      "Intervention": "Sumatriptan 50mg PO at migraine onset plus prospective headache diary tracking.",
      "Comparison_Alternative": "Targeted serotonin 5-HT1B/1D agonist therapy (Sumatriptan 50mg) versus over-the-counter NSAID monotherapy (Ibuprofen).",
      "Outcome": "Rapid abortion of acute migraine episodes within 2 hours and evaluation of attack frequency at 1 month."},
     {"CC": "Recurrent unilateral pulsatile headaches preceded by visual aura (3 episodes this month).",
      "HPI": "Patient reports right-sided pulsatile cephalalgia preceded by a 20-minute scintillating scotoma, accompanied by photophobia, phonophobia, and nausea without emesis.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Cranial nerves II-XII intact; normal motor, sensory, and cerebellar coordination exams.",
      "Assessment": "Migraine with visual aura."}
    ),
    ("Neurology", "H81.10",
     "Doc, since Tuesday morning, every time I roll over onto my right side in bed, the whole bedroom spins like a merry-go-round for thirty seconds and I feel like I'm gonna hurl.",
     "Do you have any ringing in your ears, hearing loss, or double vision when the spinning happens?",
     "Ears hear totally fine, no ringing, and if I stay perfectly still the spinning stops in under a minute.",
     "Cranial nerves and gait are normal, and performing the Dix-Hallpike maneuver toward the right reproduces torsional nystagmus and vertigo. You have Benign Paroxysmal Positional Vertigo (BPPV). We will perform the canalith repositioning Epley maneuver right here in clinic today.",
     "Can you just give me motion sickness pills like Gravol to take every day instead of moving my head around?",
     "Vestibular suppressants like dimenhydrinate only mask dizziness temporarily and slow brain compensation, whereas the Epley maneuver physically moves the displaced inner ear crystals back into place with an 80% single-session cure rate.",
     {"whole bedroom spins like a merry-go-round": "rotational vertigo", "feel like I'm gonna hurl": "vertigo-induced nausea", "ringing in your ears": "tinnitus", "double vision": "diplopia"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Right-sided Benign Paroxysmal Positional Vertigo (BPPV) of the posterior semicircular canal.",
      "Intervention": "In-office Epley canalith repositioning maneuver.",
      "Comparison_Alternative": "Physical canalith repositioning (Epley maneuver) versus daily vestibular suppressant medication (dimenhydrinate / Gravol).",
      "Outcome": "Immediate mechanical clearance of canalithiasis and resolution of positional vertigo within 24–48 hours."},
     {"CC": "Brief episodes of positional rotational vertigo triggered by rolling over in bed since Tuesday.",
      "HPI": "Patient reports <60-second episodes of rotational vertigo and nausea triggered by right lateral head movement. Denies tinnitus, hearing loss, or diplopia.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Positive right-sided Dix-Hallpike test with fatigable upbeat-torsional nystagmus; cranial nerves intact.",
      "Assessment": "Right posterior canal Benign Paroxysmal Positional Vertigo (BPPV)."}
    ),
    ("Neurology", "G56.00",
     "For the last two months, I keep waking up at night shaking my right hand out because my thumb, index, and middle fingers are totally numb and tingling like electric shocks.",
     "Are you dropping coffee mugs or noticing weakness when you pinch things with your thumb?",
     "Haven't dropped anything yet, just typing all day at work makes the tingling flare up big time.",
     "Thenar muscle bulk and thumb abduction strength are normal, but tapping over your right volar wrist triggers tingling into your first three digits (positive Tinel's and Phalen's signs). You have mild-to-moderate right Carpal Tunnel Syndrome. I recommend wearing a neutral wrist splint every night and ergonomic keyboard adjustments for 6 weeks.",
     "Should I go straight for carpal tunnel surgery so I don't lose work time?",
     "In mild-to-moderate carpal tunnel without muscle wasting or weakness, nocturnal neutral wrist splinting resolves symptoms in a majority of patients without surgical recovery downtime. If symptoms persist after 6 weeks, we will order nerve conduction studies.",
     {"numb and tingling like electric shocks": "paresthesia and dysesthesia in median nerve distribution", "shaking my right hand out at night": "nocturnal flick sign"},
     ["Blood Pressure", "Heart Rate", "Allergies", "Past Surgical History"],
     {"Patient_Diagnosis": "Mild-to-moderate Right Carpal Tunnel Syndrome (median nerve entrapment at wrist).",
      "Intervention": "Nocturnal neutral-position wrist splinting and workplace ergonomic modifications for 6 weeks.",
      "Comparison_Alternative": "Conservative nocturnal wrist splinting versus immediate surgical carpal tunnel decompression in the absence of thenar atrophy.",
      "Outcome": "Resolution of nocturnal median nerve paresthesia and preservation of hand function at 6-week follow-up."},
     {"CC": "Nocturnal numbness and paresthesia in the right thumb, index, and middle fingers for 2 months.",
      "HPI": "Patient reports nocturnal paresthesia ('electric shocks') in the right median nerve distribution relieved by shaking the hand (flick sign), exacerbated by daytime typing. Denies grip weakness.",
      "Vitals": "BP: Not Documented, HR: Not Documented.",
      "Exam": "Positive Tinel's and Phalen's signs at the right wrist; intact thenar bulk and 5/5 thumb abduction strength.",
      "Assessment": "Right Carpal Tunnel Syndrome."}
    ),
    ("Neurology", "G25.81",
     "Doc, I can't get a wink of sleep because the second I lie down in bed at night, I get this creepy-crawly worm feeling deep inside both my calves and I have to get up and pace the hallway.",
     "Does walking around make the creepy-crawly feeling go away temporarily?",
     "Yes, as soon as I walk around the kitchen it stops, but the minute I get back under the covers my legs start twitching again.",
     "Neurological exam and distal pulses in both legs are normal. However, your lab work shows a low serum ferritin of 18 ug/L. You have Restless Legs Syndrome (RLS) driven by iron deficiency. I want to start oral Ferrous Sulfate 325mg with Vitamin C every other day and eliminate evening caffeine.",
     "My neighbour takes a dopamine pill called Pramipexole for restless legs—why not start me on that right away?",
     "When ferritin is below 75 ug/L, iron repletion directly corrects the root brain dopamine cofactor deficiency without the long-term 'augmentation' worsening risk seen with dopamine agonists like Pramipexole. Let's recheck ferritin and symptoms in 8 weeks.",
     {"creepy-crawly worm feeling deep inside both calves": "bilateral lower extremity dysesthesia / akathisia", "have to get up and pace": "irresistible urge to move the legs relieved by ambulation"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Allergies"],
     {"Patient_Diagnosis": "Restless Legs Syndrome (Willis-Ekbom Disease) secondary to suboptimal iron stores (ferritin 18 ug/L).",
      "Intervention": "Oral Ferrous Sulfate 325mg + Vitamin C every other day and sleep hygiene / caffeine elimination.",
      "Comparison_Alternative": "Targeted oral iron repletion therapy versus first-line dopamine agonist pharmacotherapy (Pramipexole) in iron-deficient RLS.",
      "Outcome": "Normalization of serum ferritin (>75 ug/L) and cessation of nocturnal leg dysesthesia at 8-week follow-up."},
     {"CC": "Nocturnal creepy-crawly sensation in bilateral calves with urge to move legs.",
      "HPI": "Patient reports nocturnal bilateral calf dysesthesia occurring at rest in the evening, transiently relieved by pacing. Lab review reveals low serum ferritin (18 ug/L).",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Lower extremity motor, sensory, reflexes, and pedal pulses intact bilaterally.",
      "Assessment": "Restless Legs Syndrome (RLS) associated with iron deficiency."}
    ),
    ("Neurology", "G51.0",
     "I woke up this morning and panicked when I looked in the bathroom mirror—the entire left side of my face is drooping, I can't close my left eye all the way, and water dribbles out of my mouth.",
     "Can you still move your left arm and leg normally, and can you try raising both eyebrows for me?",
     "My arms and legs feel 100% strong, no numbness anywhere else, and I got some pain behind my left ear yesterday.",
     "Your left forehead does not wrinkle and the left nasolabial fold is flattened, while limb motor strength is 5/5 throughout. Because the forehead is involved, this is a peripheral facial nerve palsy—Bell's Palsy—not a central stroke. Starting within 72 hours, we will prescribe Prednisone 60mg daily for 5 days with a 5-day taper, plus artificial tears and night eye taping.",
     "Don't I need an antiviral pill like Valacyclovir by itself instead of high-dose steroids?",
     "Clinical trials prove high-dose corticosteroids like Prednisone started within 72 hours are the single most effective treatment for facial nerve recovery, whereas antiviral monotherapy is ineffective on its own. Protect your left cornea with lubricating drops and return in 1 week.",
     {"entire left side of my face is drooping": "acute unilateral left facial nerve paresis", "can't close my left eye all the way": "left lagophthalmos", "water dribbles out of my mouth": "oral incompetence"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Acute Left Bell's Palsy (idiopathic peripheral CN VII palsy).",
      "Intervention": "Oral Prednisone (60mg daily x 5 days followed by 5-day taper) initiated within 72 hours, plus ocular lubrication and nocturnal eye patching.",
      "Comparison_Alternative": "Early oral corticosteroid regimen + corneal protection versus antiviral monotherapy (Valacyclovir alone).",
      "Outcome": "Complete recovery of left facial nerve (CN VII) motor function and prevention of exposure keratitis."},
     {"CC": "Acute onset left-sided facial droop and inability to close the left eye this morning.",
      "HPI": "Patient awoke with left facial droop, left lagophthalmos, oral incompetence, and preceding left retroauricular pain. Denies limb weakness or dysarthria.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Complete left upper and lower facial paresis (involves forehead); CN II-VI, VIII-XII intact; 5/5 motor strength in all four extremities.",
      "Assessment": "Acute left Bell's Palsy (peripheral CN VII palsy)."}
    ),

    # --- ENDOCRINOLOGY (EXP_021 to EXP_025) ---
    ("Endocrinology", "E11.9",
     "My wife forced me to come in because I'm drinking water like a fish all day and getting up four times a night to pee. Plus my vision gets blurry in the afternoons.",
     "Any unexplained weight loss or numbness and tingling in your toes?",
     "Actually, yeah, my feet feel like they have pins and needles at night. No weight loss though.",
     "Clinic blood work from yesterday shows an HbA1c of 8.4% and fasting glucose of 10.2 mmol/L. Blood pressure is 132/80 mmHg. You have Type 2 Diabetes with early peripheral neuropathy. We need to start Metformin 500mg twice daily with meals.",
     "Can I try fixing this strictly with a keto diet for six months without any medication?",
     "Because your HbA1c is already at 8.4% and you have nerve symptoms in your feet, diet changes alone carry a higher risk of prolonged hyperglycemia compared to combining nutrition with first-line Metformin. Let's start both and recheck HbA1c in 3 months.",
     {"drinking water like a fish": "polydipsia", "getting up four times a night to pee": "nocturia / polyuria", "pins and needles in feet": "bilateral pedal paresthesia (diabetic peripheral neuropathy)"},
     ["Heart Rate", "Respiratory Rate", "Medication Allergies", "Surgical History"],
     {"Patient_Diagnosis": "New-onset Type 2 Diabetes Mellitus (HbA1c 8.4%) with early diabetic peripheral neuropathy.",
      "Intervention": "Metformin 500mg PO BID with meals combined with low-glycemic nutrition counseling.",
      "Comparison_Alternative": "Combined first-line Metformin + dietary therapy versus dietary modification (ketogenic diet) monotherapy.",
      "Outcome": "Glycemic control (target HbA1c <7.0%) and stabilization of pedal neuropathy at 3-month follow-up."},
     {"CC": "Polydipsia, nocturia/polyuria, afternoon blurred vision, and nocturnal pedal paresthesia.",
      "HPI": "Patient reports persistent polydipsia, nocturia (4x/night), intermittent blurred vision, and bilateral pedal paresthesia. Denies weight loss.",
      "Vitals": "BP 132/80 mmHg, HR: Not Documented, RR: Not Documented.",
      "Exam": "Decreased bilateral distal monofilament sensation; Labs: HbA1c 8.4%, Fasting Glucose 10.2 mmol/L.",
      "Assessment": "Type 2 Diabetes Mellitus with diabetic peripheral neuropathy."}
    ),
    ("Endocrinology", "E03.9",
     "Doc, over the last six months I feel like my battery is permanently drained—I'm freezing cold even when the house is at 23 degrees, my hair is thinning, and I've gained fifteen pounds despite barely eating.",
     "Have your bowel movements been sluggish, or have you noticed dry skin?",
     "Super constipated, and the skin on my shins is dry like sandpaper.",
     "Heart rate is 56 bpm, BP 118/76 mmHg. Thyroid gland is non-goitrous, and ankle reflexes show delayed relaxation. Your blood panel shows an elevated TSH of 18.5 mIU/L and low Free T4 of 7.2 pmol/L. You have primary hypothyroidism. I am prescribing Levothyroxine 75 mcg taken every morning on an empty stomach.",
     "I read online about natural desiccated pig thyroid extract and kelp iodine pills—can I take those instead of synthetic Levothyroxine?",
     "Kelp supplements can worsen autoimmune thyroiditis and desiccated animal thyroid has unpredictable hormone ratios that cause heart palpitations, whereas synthetic Levothyroxine provides exact, stable T4 replacement. We will recheck your TSH in 6 weeks.",
     {"battery is permanently drained": "chronic fatigue / lethargy", "freezing cold even when the house is warm": "cold intolerance", "hair is thinning": "diffuse alopecia", "skin dry like sandpaper": "xerosis cutis"},
     ["Temperature", "Respiratory Rate", "Allergies", "Family History"],
     {"Patient_Diagnosis": "Overt Primary Hypothyroidism (TSH 18.5 mIU/L, low Free T4).",
      "Intervention": "Levothyroxine 75 mcg PO daily on an empty stomach with 6-week TSH monitoring.",
      "Comparison_Alternative": "Standardized synthetic L-thyroxine (Levothyroxine) replacement versus desiccated animal thyroid extract or kelp iodine supplements.",
      "Outcome": "Normalization of serum TSH (0.5–4.5 mIU/L) and resolution of fatigue, cold intolerance, and constipation at 6 weeks."},
     {"CC": "Chronic fatigue, cold intolerance, 15-lb weight gain, alopecia, and constipation for 6 months.",
      "HPI": "Patient reports progressive lethargy, cold intolerance, weight gain despite low appetite, constipation, diffuse hair thinning, and xerosis.",
      "Vitals": "HR 56 bpm (bradycardia), BP 118/76 mmHg, Temp: Not Documented, RR: Not Documented.",
      "Exam": "Delayed relaxation phase of Achilles deep tendon reflexes; dry skin. Labs: TSH 18.5 mIU/L, Free T4 7.2 pmol/L.",
      "Assessment": "Primary Hypothyroidism."}
    ),
    ("Endocrinology", "E05.90",
     "I don't know what's wrong with me, Doc. My hands won't stop trembling when I hold a teacup, I'm sweating buckets even in the winter cold, and I lost twelve pounds this month even though I'm eating like a horse.",
     "Have you noticed your heart racing, feeling anxious, or your eyes feeling gritty or bulging?",
     "My heart pounds like crazy, I feel wired and anxious 24/7, and my eyes feel super dry and starey.",
     "Heart rate is 114 bpm, BP 136/74 mmHg. You have a fine resting tremor, mild bilateral proptosis, and a diffusely enlarged thyroid gland. Labs show undetectable TSH (<0.01 mIU/L), high Free T4 (38 pmol/L), and positive TSH-receptor antibodies confirming Graves' disease. We are starting Methimazole 15mg daily and Propranolol 20mg three times daily.",
     "Do I have to drink radioactive iodine right now to destroy my thyroid?",
     "Oral antithyroid medication with Methimazole plus Propranolol safely restores normal hormone levels and achieves remission in many patients without permanent thyroid destruction or radiation precautions, making it our preferred first step over immediate radioactive iodine ablation.",
     {"hands won't stop trembling": "fine resting postural tremor", "sweating buckets in the winter cold": "diaphoresis and heat intolerance", "lost twelve pounds eating like a horse": "weight loss despite hyperphagia", "eyes feel starey/bulging": "Graves' ophthalmopathy / proptosis"},
     ["Temperature", "Respiratory Rate", "Medication Allergies", "Smoking History"],
     {"Patient_Diagnosis": "Graves' Disease (Autoimmune Hyperthyroidism) with mild Graves' ophthalmopathy.",
      "Intervention": "Methimazole 15mg PO daily (antithyroid therapy) plus Propranolol 20mg PO TID (adrenergic symptom control).",
      "Comparison_Alternative": "First-line oral thionamide therapy (Methimazole) + beta-blockade versus immediate Radioactive Iodine (RAI) thyroid ablation.",
      "Outcome": "Achieve biochemical euthyroidism, control resting tachycardia/tremor, and monitor CBC/liver enzymes at 4 weeks."},
     {"CC": "Tremor, heat intolerance, diaphoresis, palpitations, and 12-lb weight loss despite hyperphagia.",
      "HPI": "Patient reports 1-month history of postural hand tremor, heat intolerance, anxiety, palpitations, weight loss with increased appetite, and ocular dryness.",
      "Vitals": "HR 114 bpm (sinus tachycardia), BP 136/74 mmHg, Temp: Not Documented, RR: Not Documented.",
      "Exam": "Fine outstretched hand tremor, diffuse non-tender thyromegaly, mild bilateral exophthalmos/proptosis. Labs: TSH <0.01, Free T4 38 pmol/L, TRAb positive.",
      "Assessment": "Graves' Hyperthyroidism."}
    ),
    ("Endocrinology", "M10.9",
     "Doc, I woke up at 4 AM screaming because the base of my right big toe is swollen up like a ripe plum, glowing red-hot, and hurts so bad I couldn't even let the bedsheet touch it!",
     "Did you have a big steak dinner or a few beers over the weekend, or take any water pills?",
     "Yeah, we celebrated at the steakhouse on Saturday night and I had four IPAs and shellfish.",
     "Temperature is 37.3°C. Your right first metatarsophalangeal (MTP) joint is erythematous, warm, swollen, and exquisitely tender to light touch. This is classic acute gouty arthritis (podagra). Because your kidney function is normal, I'm prescribing high-dose Naproxen 500mg twice daily (or low-dose Colchicine) for 5 days.",
     "My uncle takes Allopurinol every day for gout—should I start taking Allopurinol today to make this swelling go down?",
     "Starting uric-acid lowering Allopurinol during an acute gout attack actually destabilizes joint crystals and makes the flare last longer, whereas acute anti-inflammatory therapy (NSAIDs/Colchicine) resolves the pain first before we consider preventive Allopurinol later.",
     {"base of my right big toe is swollen like a ripe plum and glowing red-hot": "acute first metatarsophalangeal (MTP) monoarthritis (podagra)", "couldn't even let the bedsheet touch it": "extreme allodynia / tenderness"},
     ["Blood Pressure", "Heart Rate", "Medication Allergies", "Family History"],
     {"Patient_Diagnosis": "Acute Gouty Arthritis of the right first MTP joint (Podagra).",
      "Intervention": "Acute anti-inflammatory regimen (Naproxen 500mg PO BID for 5 days), joint rest, ice, and purine/alcohol restriction.",
      "Comparison_Alternative": "Acute anti-inflammatory therapy (NSAID/Colchicine) during flare versus de novo initiation of urate-lowering therapy (Allopurinol) during an acute attack.",
      "Outcome": "Complete resolution of acute MTP joint inflammation and pain within 5–7 days."},
     {"CC": "Acute severe pain, swelling, and erythema of the right first MTP joint since 4 AM.",
      "HPI": "Patient awoke with severe right first MTP joint pain, swelling, erythema, and allodynia following high-purine/alcohol intake. Denies prior episodes.",
      "Vitals": "Temp 37.3°C, BP: Not Documented, HR: Not Documented.",
      "Exam": "Marked erythema, warmth, edema, and tenderness isolated to the right 1st MTP joint.",
      "Assessment": "Acute gouty monoarthritis (podagra)."}
    ),
    ("Endocrinology", "M81.0",
     "I came in to review that bone density scan you ordered after I slipped on the porch steps and cracked my wrist two months ago.",
     "Are you currently taking any calcium or vitamin D supplements, and did either of your parents ever break a hip?",
     "My mom broke her hip at seventy-two. I take a tiny vitamin D gummy sometimes when I remember.",
     "Your DEXA bone mineral density scan shows a lumbar spine T-score of -2.8 and femoral neck T-score of -2.6, with normal blood calcium and kidney function. Combined with your fragility wrist fracture, this confirms postmenopausal osteoporosis. I recommend starting weekly Alendronate 70mg plus daily Calcium 1200mg and Vitamin D 2000 IU.",
     "Can I just take high-dose calcium and vitamin D pills without taking the weekly Alendronate bone medication?",
     "Calcium and Vitamin D are necessary building blocks, but in established osteoporosis with a T-score of -2.8 and prior wrist fracture, supplements alone do not stop bone breakdown, whereas a bisphosphonate like Alendronate cuts hip and spine fracture risk by up to 50%.",
     {"cracked my wrist from a slip": "fragility distal radius (Colles') fracture", "bone density scan": "dual-energy X-ray absorptiometry (DEXA)"},
     ["Blood Pressure", "Heart Rate", "Allergies", "Smoking History"],
     {"Patient_Diagnosis": "Postmenopausal Osteoporosis (Spine T-score -2.8) with history of fragility wrist fracture.",
      "Intervention": "Alendronate 70mg PO once weekly plus daily elemental Calcium (1200mg) and Vitamin D3 (2000 IU) supplementation.",
      "Comparison_Alternative": "Antiresorptive bisphosphonate therapy (Alendronate) + Calcium/Vitamin D versus Calcium and Vitamin D supplementation monotherapy.",
      "Outcome": "Stabilization/improvement of bone mineral density on 2-year repeat DEXA and prevention of vertebral/hip fragility fractures."},
     {"CC": "Follow-up of DEXA bone mineral density scan following fragility wrist fracture.",
      "HPI": "Patient with recent low-trauma wrist fracture and maternal history of hip fracture presents for DEXA review. Inconsistent Vitamin D supplementation.",
      "Vitals": "BP: Not Documented, HR: Not Documented.",
      "Exam": "DEXA: Lumbar spine T-score -2.8, Femoral neck T-score -2.6; serum calcium and eGFR normal.",
      "Assessment": "Severe Postmenopausal Osteoporosis."}
    ),

    # --- ORTHOPEDICS / MSK (EXP_026 to EXP_030) ---
    ("Orthopedics", "M54.50",
     "Doc, I lifted a heavy bag of water softener salt out of my trunk on Sunday and felt a pull in my lower back. It's super stiff and sore right across my beltline.",
     "Does the pain shoot down past your knee into your foot, or do you have any numbness in your groin or trouble controlling your bladder?",
     "No pain down my legs at all, no numbness, and bathroom stuff is totally normal.",
     "Gait is slightly guarded, lumbar paraspinal muscles are tender and tight, but straight-leg raise is negative bilaterally and lower limb strength and reflexes are 5/5. You have an acute mechanical lumbar muscle strain with zero red flags. Stay active with gentle walking, apply heat, and use short-term Naproxen.",
     "Shouldn't I stay flat in bed for a week and get a lumbar spine X-ray or MRI today?",
     "Prolonged bed rest actually stiffens muscles and delays recovery compared to staying gently active, and routine lumbar imaging in the first 6 weeks without red flags does not improve outcomes and exposes you to unnecessary radiation.",
     {"felt a pull right across my beltline": "acute mechanical lumbosacral strain", "pain shooting down past the knee": "lumbar radiculopathy / sciatica", "numbness in the groin": "saddle anesthesia"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Acute Mechanical Low Back Pain (Lumbar Paraspinal Strain) without radiculopathy or cauda equina red flags.",
      "Intervention": "Early mobilization (avoiding bed rest), superficial heat therapy, and short-course oral NSAID (Naproxen) therapy.",
      "Comparison_Alternative": "Active conservative management without immediate imaging versus strict bed rest and immediate lumbar spine X-ray/MRI.",
      "Outcome": "Restoration of full lumbar range of motion and pain resolution within 2–4 weeks."},
     {"CC": "Acute low back pain and stiffness following heavy lifting 3 days ago.",
      "HPI": "Patient reports lumbosacral muscular pain and stiffness onset after lifting heavy salt bag. Denies radicular leg pain, paresthesia, saddle anesthesia, or bowel/bladder dysfunction.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Bilateral lumbar paraspinal tenderness; negative straight-leg raise; 5/5 lower extremity motor strength and intact reflexes.",
      "Assessment": "Acute mechanical lumbar strain."}
    ),
    ("Orthopedics", "M17.12",
     "I've been having a lot of pain in my left knee for the last three months. It's really stiff when I get out of bed for about twenty minutes, and climbing stairs makes it grind and ache.",
     "Did you twist or injure it recently, or is the knee hot and red like an infection?",
     "No injury, just gradually got worse over the years. It puffs up a tiny bit after a long walk, not red or hot.",
     "Left knee exam shows mild bony enlargement, crepitus on flexion, no warmth or erythema, and stable ligaments (negative Lachman and McMurray tests). Weight-bearing X-ray shows medial joint space narrowing and small osteophytes. You have primary osteoarthritis of the left knee. I recommend physical therapy for quadriceps strengthening plus topical Diclofenac gel.",
     "Can you give me an arthroscopic 'clean-out' knee surgery or strong opioid pills like Oxycodone for the stair pain?",
     "Arthroscopic lavage and opioids provide no long-term benefit for degenerative knee osteoarthritis and carry high complication risks, whereas quadriceps strengthening physical therapy and topical NSAIDs safely reduce pain and improve joint mechanics.",
     {"stiff when I get out of bed for twenty minutes": "brief morning articular stiffness (<30 minutes)", "grind and ache on stairs": "patellofemoral/tibiofemoral crepitus and mechanical arthralgia"},
     ["Blood Pressure", "Heart Rate", "Allergies", "Past Surgical History"],
     {"Patient_Diagnosis": "Primary Osteoarthritis of the Left Knee.",
      "Intervention": "Structured physical therapy (quadriceps/hip strengthening) combined with topical Diclofenac 2% gel QID.",
      "Comparison_Alternative": "Targeted physical therapy + topical NSAID therapy versus arthroscopic debridement surgery or oral opioid analgesics.",
      "Outcome": "Improved knee functional mobility on stairs and reduced mechanical joint pain at 8-week follow-up."},
     {"CC": "Chronic left knee mechanical pain, morning stiffness, and crepitus for 3 months.",
      "HPI": "Patient reports insidious onset left knee pain exacerbated by stairs and weight-bearing, with <20 minutes of morning stiffness. Denies acute trauma or systemic symptoms.",
      "Vitals": "BP: Not Documented, HR: Not Documented.",
      "Exam": "Left knee crepitus, mild medial joint line tenderness, no erythema; negative Lachman/McMurray tests. X-ray: medial joint space narrowing and osteophytosis.",
      "Assessment": "Left knee primary osteoarthritis."}
    ),
    ("Orthopedics", "M75.101",
     "Doc, for six weeks my right shoulder has had a deep dull ache on the outside of my arm, and it kills me whenever I reach up to the top kitchen cupboard or try to sleep on my right side.",
     "Did you fall on your outstretched arm, or does your neck hurt when you turn your head?",
     "No fall at all, neck feels totally fine, just painted the bedroom ceiling a month ago.",
     "Cervical spine range of motion is full and painless. Right shoulder shows a painful arc between 60 and 120 degrees of abduction, positive Hawkins-Kennedy impingement sign, and 4+/5 supraspinatus strength without a drop-arm sign. This is rotator cuff tendinopathy with subacromial impingement. Let's start a 6-week rotator cuff physiotherapy program and anti-inflammatory icing.",
     "Should we inject cortisone into my shoulder three times right now or book rotator cuff surgery?",
     "In non-traumatic rotator cuff tendinopathy without a full-thickness tear, structured scapular and rotator cuff rehabilitation achieves identical long-term outcomes to surgery without tendon weakening from repeated steroid injections.",
     {"ache on the outside of my arm reaching overhead": "deltoid-referred subacromial impingement pain", "kills me trying to sleep on my right side": "nocturnal lateral decubitus shoulder pain"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Right Subacromial Impingement Syndrome / Rotator Cuff Tendinopathy.",
      "Intervention": "6-week supervised rotator cuff and scapular stabilization physiotherapy program plus activity modification.",
      "Comparison_Alternative": "Structured physical therapy rehabilitation versus immediate surgical subacromial decompression or repeated corticosteroid injections.",
      "Outcome": "Pain-free overhead abduction (>150 degrees) and restoration of nocturnal comfort at 6-week follow-up."},
     {"CC": "Right shoulder overhead pain and nocturnal pain for 6 weeks.",
      "HPI": "Patient reports lateral right shoulder pain triggered by overhead reaching and lying on the affected side following ceiling painting. Denies trauma or cervical pain.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Positive painful arc (60–120°), positive Hawkins-Kennedy test, negative drop-arm test; intact cervical ROM.",
      "Assessment": "Right rotator cuff tendinopathy with subacromial impingement."}
    ),
    ("Orthopedics", "M72.2",
     "Every single morning when I swing my legs out of bed and take my first step on the hardwood floor, it feels like I stepped barefoot on a sharp thumbtack right under my left heel.",
     "Does the heel pain loosen up a bit after you walk around for ten minutes, and then come back at the end of a long shift?",
     "That's exactly it! Better after ten minutes, then throbs again by 4 PM.",
     "Palpation reveals focal tenderness right at the medial calcaneal tubercle of the left foot, aggravated by passive dorsiflexion of the big toe (positive Windlass test). Ankle motion and Achilles tendon are intact. You have left plantar fasciitis. I'm prescribing daily plantar fascia and calf stretching, supportive heel orthotics, and rolling a frozen water bottle under your arch.",
     "Do I need a heel X-ray to check for a bone spur and have the spur shaved off surgically?",
     "Heel bone spurs on X-ray are incidental reactions rather than the cause of pain, and plantar fasciastretching plus arch orthotics resolves 90% of plantar fasciitis cases without invasive surgery.",
     {"stepped barefoot on a sharp thumbtack on first step in the morning": "post-static dyskinesia (first-step plantar heel pain)"},
     ["Blood Pressure", "Heart Rate", "Allergies", "Family History"],
     {"Patient_Diagnosis": "Left Plantar Fasciitis.",
      "Intervention": "Plantar fascia-specific stretching protocol, gastrocnemius-soleus stretching, supportive heel cup orthotics, and cryotherapy.",
      "Comparison_Alternative": "Conservative biomechanical stretching and orthotic therapy versus diagnostic calcaneal X-ray and surgical heel spur excision.",
      "Outcome": "Resolution of morning first-step calcaneal pain and return to prolonged standing within 6–8 weeks."},
     {"CC": "Sharp left plantar heel pain upon first morning weight-bearing.",
      "HPI": "Patient describes severe sharp left medial heel pain on initial morning steps (post-static dyskinesia) that improves with brief ambulation and recurs after prolonged standing.",
      "Vitals": "BP: Not Documented, HR: Not Documented.",
      "Exam": "Point tenderness at the left medial calcaneal tuberosity; positive Windlass test; intact Achilles tendon.",
      "Assessment": "Left plantar fasciitis."}
    ),
    ("Orthopedics", "S93.401A",
     "I stepped in a pothole walking the dog yesterday evening and rolled my right ankle inward really hard. The outside of my ankle swelled up like a golf ball.",
     "Were you able to walk four steps on it right after it happened and walk into the clinic today?",
     "Yeah, I limped all the way home on it and walked in here today, just sore on the outer ankle bone.",
     "Right ankle shows mild soft tissue swelling and tenderness over the anterior talofibular ligament (ATFL), but zero bone tenderness along the posterior edge or tip of either malleolus, the navicular, or the base of the 5th metatarsal. By the Ottawa Ankle Rules, no fracture X-ray is required. This is a Grade I/II lateral ankle sprain. Use a lace-up ankle brace, RICE protocol, and early gentle range-of-motion exercises.",
     "Shouldn't you put my leg in a hard plaster cast for four weeks and X-ray it just in case?",
     "The Ottawa Ankle Rules have nearly 100% sensitivity for ruling out fractures without X-rays, and functional bracing with early gentle movement heals ankle ligaments faster and prevents stiffness compared to rigid plaster casting.",
     {"rolled my right ankle inward": "inversion injury of the right ankle", "swelled up like a golf ball on the outside": "lateral perimalleolar soft tissue edema"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Acute Grade I/II Right Lateral Ankle Sprain (ATFL injury), Ottawa Ankle Rules negative.",
      "Intervention": "Functional lace-up ankle bracing, Rest/Ice/Compression/Elevation (RICE), and early progressive weight-bearing/ROM exercises.",
      "Comparison_Alternative": "Functional ankle bracing + Ottawa Ankle Rule clinical clearance versus unnecessary ankle radiography and rigid plaster cast immobilization.",
      "Outcome": "Resolution of lateral ligamentous swelling and return to full stable ambulation within 2–3 weeks."},
     {"CC": "Right lateral ankle pain and swelling following an inversion injury yesterday.",
      "HPI": "Patient sustained an inversion injury to the right ankle stepping in a pothole. Able to bear weight immediately and in clinic today.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Tenderness and mild edema localized to the right ATFL; zero bony tenderness at lateral/medial malleoli, navicular, or 5th metatarsal base (Ottawa Ankle Rules negative).",
      "Assessment": "Acute right lateral ankle sprain (ATFL)."}
    ),

    # --- DERMATOLOGY (EXP_031 to EXP_035) ---
    ("Dermatology", "L20.9",
     "Doc, every autumn when the furnace kicks on, the insides of my elbows and behind my knees break out in these angry red, scaly patches that itch so bad I scratch them raw in my sleep.",
     "Do you have a history of asthma or seasonal hay fever, and what soap or lotion are you using right now?",
     "Had hay fever since I was a kid. I've been scrubbing it with scented antibacterial soap because I thought it was dirty.",
     "Skin exam shows erythematous, lichenified, excoriated plaques in the bilateral antecubital and popliteal fossae without honey-colored crusting or pustules. This is an atopic dermatitis (eczema) flare triggered by dry indoor air and harsh soap. Stop the scented soap, apply thick fragrance-free petrolatum/ceramide ointment immediately after lukewarm showers, and use Triamcinolone 0.1% cream twice daily for 14 days.",
     "Can I just keep putting over-the-counter antibiotic Polysporin ointment on it instead of a steroid cream?",
     "Topical antibiotics like Polysporin do not treat eczema inflammation and frequently cause allergic contact dermatitis, whereas a medium-potency topical corticosteroid like Triamcinolone calms the immune flare while barrier ointments lock in moisture.",
     {"angry red, scaly patches that itch so bad I scratch them raw": "pruritic erythematous lichenified plaques with excoriations", "insides of my elbows and behind my knees": "flexural distribution (antecubital and popliteal fossae)", "hay fever": "allergic rhinitis"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Moderate Atopic Dermatitis (Flexural Eczema) flare without secondary bacterial superinfection.",
      "Intervention": "Triamcinolone acetonide 0.1% cream BID x 14 days combined with fragrance-free ceramide/petrolatum emollient barrier therapy and gentle cleanser substitution.",
      "Comparison_Alternative": "Topical corticosteroid + barrier emollient regimen versus OTC topical antibiotic ointment (Polysporin) and harsh antibacterial soaps.",
      "Outcome": "Cessation of nocturnal pruritus, resolution of flexural erythema/excoriations, and restoration of epidermal barrier at 2 weeks."},
     {"CC": "Intensely pruritic red scaly rash in bilateral elbow and knee creases.",
      "HPI": "Patient with history of allergic rhinitis presents with seasonal flare of flexural pruritic rash exacerbated by scented antibacterial soap. Denies purulent drainage.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Erythematous, xerotic, lichenified plaques with excoriations in bilateral antecubital and popliteal fossae; no impetiginization.",
      "Assessment": "Atopic dermatitis flare."}
    ),
    ("Dermatology", "B02.9",
     "Four days ago I started getting this weird burning electric tingling along the left side of my ribs, and yesterday morning a strip of angry red blisters popped up right along that exact line.",
     "Does the rash cross over the middle of your stomach to the right side, and is it anywhere near your face or eyes?",
     "Nope, stops dead center at my belly button and wraps around my left back. Nowhere near my face.",
     "Vital signs show temperature 37.1°C. Exam reveals grouped vesicles on an erythematous base restricted Strictly to the left T8 dermatome without crossing the midline. You have Herpes Zoster (shingles). We are starting oral Valacyclovir 1000mg three times daily for 7 days plus Gabapentin / acetaminophen for acute nerve pain.",
     "Can I just put calamine lotion on the blisters and let it run its course without taking antiviral pills?",
     "Calamine only cools the surface skin, whereas oral Valacyclovir stops viral replication, speeds blister crusting, and significantly lowers your risk of developing months of debilitating postherpetic neuralgia nerve pain.",
     {"burning electric tingling along my ribs": "prodromal dermatomal neuropathic pain", "strip of angry red blisters": "grouped vesicular eruption on an erythematous base in a unilateral dermatomal distribution"},
     ["Blood Pressure", "Heart Rate", "Medication Allergies", "Immunization History"],
     {"Patient_Diagnosis": "Acute Herpes Zoster (Shingles) of the left T8 dermatome.",
      "Intervention": "Valacyclovir 1000mg PO TID for 7 days plus multimodal neuropathic analgesia and lesion isolation precautions until crusted.",
      "Comparison_Alternative": "Systemic oral antiviral therapy (Valacyclovir) versus topical symptomatic calamine lotion monotherapy.",
      "Outcome": "Accelerated vesicular crusting, acute pain resolution, and reduction of postherpetic neuralgia (PHN) incidence."},
     {"CC": "Painful unilateral blister rash along the left torso for 24 hours, preceded by 4 days of burning pain.",
      "HPI": "Patient reports prodromal burning pain along the left lower thorax followed by a unilateral vesicular rash that does not cross the midline. Denies ocular involvement.",
      "Vitals": "Temp 37.1°C, BP: Not Documented, HR: Not Documented.",
      "Exam": "Grouped vesicles on an erythematous base strictly confined to the left T8 dermatome.",
      "Assessment": "Acute Herpes Zoster (left T8 dermatome)."}
    ),
    ("Dermatology", "L71.9",
     "Doc, I'm so embarrassed—every time I drink a cup of hot tea or walk in from the cold wind, my cheeks and nose turn bright lobster red and stay flushed for hours, with tiny red pimples.",
     "Do you see any blackheads on your cheeks, or do your eyes ever feel gritty and bloodshot?",
     "No blackheads at all, and my eyes feel totally normal.",
     "Facial skin exam shows central facial erythema over the malar cheeks and nasal bridge with telangiectasias and scattered inflammatory papules, but zero comedones. Absence of comedones rules out acne vulgaris and confirms papulopustular Rosacea. I'm prescribing topical Metronidazole 0.75% gel daily, daily mineral SPF 50 sunscreen, and trigger avoidance.",
     "I bought a harsh salicylic acid acne scrub and benzoyl peroxide wash—should I scrub my cheeks with that twice a day?",
     "Abrasive acne scrubs and benzoyl peroxide strip the fragile skin barrier in rosacea and trigger severe burning flares, whereas gentle anti-inflammatory topical Metronidazole and mineral sunscreen calm vascular reactivity.",
     {"cheeks and nose turn bright lobster red": "central facial flushing and persistent centrofacial erythema", "tiny red pimples with broken blood vessels": "inflammatory papules and telangiectasias"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Allergies"],
     {"Patient_Diagnosis": "Papulopustular Rosacea.",
      "Intervention": "Topical Metronidazole 0.75% gel daily, broad-spectrum mineral sunscreen (SPF 50), gentle non-soap cleanser, and thermal trigger avoidance.",
      "Comparison_Alternative": "Targeted anti-inflammatory rosacea therapy (topical Metronidazole + barrier protection) versus abrasive OTC acne scrubs (salicylic acid / benzoyl peroxide).",
      "Outcome": "Reduction in inflammatory facial papules and attenuation of centrofacial flushing at 6-week follow-up."},
     {"CC": "Persistent central facial redness, flushing triggered by heat/wind, and facial papules.",
      "HPI": "Patient reports prolonged malar and nasal flushing triggered by hot beverages and cold wind, accompanied by small red papules. Denies ocular symptoms.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Centrofacial erythema, telangiectasias, and inflammatory papules over cheeks and nose; complete absence of comedones.",
      "Assessment": "Papulopustular Rosacea."}
    ),
    ("Dermatology", "L03.115",
     "Look at my right shin, Doc. I scraped it on a rusty garden rake three days ago, and since yesterday the redness has spread halfway up my calf, it's hot like a radiator, and throbbing.",
     "Have you had any fever, chills, or pus draining out of a big boil?",
     "Felt chilly this morning, no pus pocket or boil, just solid spreading red skin.",
     "Temperature is 38.1°C, heart rate 90 bpm. Your right anterior tibia shows a 10x12 cm poorly demarcated area of spreading erythema, warmth, edema, and tenderness without fluctuance or crepitus. I've outlined the border with a surgical pen. You have non-purulent acute bacterial cellulitis. I'm prescribing Cephalexin 500mg four times daily for 5 to 7 days and leg elevation.",
     "Do you need to cut my leg open with a scalpel right now to drain it?",
     "Incision and drainage is only helpful when there is a walled-off pus collection (abscess), whereas diffuse non-purulent cellulitis lives inside the skin tissue layers and requires systemic oral Cephalexin antibiotics plus elevation. Return in 48 hours to ensure the redness recedes inside the pen line.",
     {"redness spread halfway up my calf and hot like a radiator": "expanding lower extremity erythema, calor, and induration", "pus pocket or boil": "fluctuant cutaneous abscess"},
     ["Blood Pressure", "Respiratory Rate", "Medication Allergies", "Tetanus Status"],
     {"Patient_Diagnosis": "Acute Non-Purulent Bacterial Cellulitis of the right lower extremity.",
      "Intervention": "Oral Cephalexin 500mg QID x 5–7 days, demarcation of erythema borders, and limb elevation.",
      "Comparison_Alternative": "Systemic oral beta-lactam antibiotic therapy (Cephalexin) versus surgical incision and drainage (indicated only for fluctuant abscesses).",
      "Outcome": "Defervescence and regression of erythema within demarcated pen borders at 48-hour recheck."},
     {"CC": "Spreading redness, warmth, swelling, and throbbing pain on the right shin for 24 hours.",
      "HPI": "Patient sustained a minor abrasion on the right shin 3 days ago, followed by rapidly expanding erythema, warmth, and chills. Denies purulent drainage.",
      "Vitals": "Temp 38.1°C, HR 90 bpm, BP: Not Documented, RR: Not Documented.",
      "Exam": "10x12 cm area of confluent erythema, warmth, tenderness, and edema on right anterior leg; no fluctuance or crepitus. Borders marked.",
      "Assessment": "Acute non-purulent cellulitis of the right lower leg."}
    ),
    ("Dermatology", "L40.0",
     "For the last few months I've had these thick, raised red patches covered in silvery-white flaky scales on the outside of both my elbows and knees.",
     "Do any of your fingers or toes swell up like little sausages, or do your fingernails have tiny pits in them?",
     "My nails have a few tiny pinprick dents, yeah, but my joints don't hurt or swell at all.",
     "Skin exam reveals well-demarcated erythematous plaques with thick silvery micaceous scale over the bilateral extensor elbows and knees (positive Auspitz sign when scraped), covering about 4% of body surface area, plus nail pitting. No joint tenderness. This is mild-to-moderate chronic plaque psoriasis. Let's start combination Calcipotriol-Betamethasone dipropionate ointment once daily for 4 weeks.",
     "Should I go straight onto those injectable biologic shots I see advertised on TV?",
     "Systemic biologic injections are reserved for severe psoriasis covering over 10% of the body or psoriatic arthritis, whereas localized plaque psoriasis (<5% BSA) responds safely and rapidly to targeted topical Vitamin D analog plus corticosteroid ointment.",
     {"thick raised red patches covered in silvery-white flaky scales": "well-demarcated erythematous plaques with micaceous silvery scale", "outside of both elbows and knees": "bilateral extensor surface distribution", "tiny pinprick dents in fingernails": "ungual pitting"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Family History"],
     {"Patient_Diagnosis": "Mild-to-Moderate Chronic Plaque Psoriasis (<5% BSA) with nail pitting, without psoriatic arthritis.",
      "Intervention": "Topical Calcipotriol / Betamethasone dipropionate ointment applied once daily to extensor plaques for 4 weeks.",
      "Comparison_Alternative": "First-line fixed-dose topical Vitamin D analog + corticosteroid therapy versus systemic biologic immunotherapy for limited (<5% BSA) disease.",
      "Outcome": "Clearance of hyperkeratotic silvery scale and flattening of extensor plaques at 4-week follow-up."},
     {"CC": "Thick red scaly plaques on bilateral elbows and knees for several months.",
      "HPI": "Patient presents with well-demarcated scaly plaques on extensor surfaces and fingernail pitting. Denies arthralgia or dactylitis.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Erythematous plaques with silvery micaceous scale on bilateral extensor elbows and knees (~4% BSA); mild nail pitting; no joint synovitis.",
      "Assessment": "Chronic plaque psoriasis."}
    ),

    # --- PSYCHIATRY / MENTAL HEALTH (EXP_036 to EXP_040) ---
    ("Psychiatry", "F41.1",
     "Doc, for the past seven months my brain feels like a browser with fifty tabs open at once—I worry constantly about work, money, and family, my shoulders are knotted up, and I lie awake staring at the ceiling.",
     "Do you ever have sudden out-of-the-blue panic attacks, or any thoughts of hurting yourself?",
     "No panic attacks and definitely no thoughts of hurting myself, just non-stop background dread every single day.",
     "Blood pressure is 128/80 mmHg, pulse 78 bpm, and your thyroid panel and CBC are completely normal. Your GAD-7 score today is 16 (severe generalized anxiety). You meet criteria for Generalized Anxiety Disorder (GAD). I recommend starting Escitalopram 10mg daily combined with a referral for Cognitive Behavioral Therapy (CBT).",
     "Can you just give me a bottle of Ativan (lorazepam) to take every morning instead of an antidepressant that takes weeks to kick in?",
     "Daily benzodiazepines like Lorazepam cause tolerance, daytime sedation, and dependence within weeks, whereas an SSRI like Escitalopram paired with CBT safely rewires anxious pathways for durable long-term relief. Let's check in at 2 weeks.",
     {"brain feels like fifty tabs open worrying constantly": "pervasive excessive and uncontrollable worry", "shoulders are knotted up": "somatic muscle tension", "lie awake staring at the ceiling": "sleep-onset insomnia"},
     ["Temperature", "Respiratory Rate", "Medication Allergies", "Substance Use History"],
     {"Patient_Diagnosis": "Generalized Anxiety Disorder (GAD), GAD-7 score 16, with normal thyroid screening.",
      "Intervention": "Escitalopram 10mg PO daily (starting at 5mg x 1 week) combined with structured Cognitive Behavioral Therapy (CBT).",
      "Comparison_Alternative": "First-line SSRI pharmacotherapy + CBT versus chronic daily benzodiazepine (Lorazepam) monotherapy.",
      "Outcome": "50% reduction in GAD-7 score (<8), resolution of somatic tension, and improved sleep initiation by 6–8 weeks."},
     {"CC": "Persistent excessive worry, muscle tension, and sleep-onset insomnia for 7 months.",
      "HPI": "Patient reports daily uncontrollable worry across multiple domains, somatic shoulder tension, and insomnia for 7 months. GAD-7 score = 16. Denies panic attacks or suicidal ideation.",
      "Vitals": "BP 128/80 mmHg, HR 78 bpm, Temp: Not Documented, RR: Not Documented.",
      "Exam": "Alert, cooperative, anxious affect, linear thought process, intact insight/judgment. Normal TSH and CBC.",
      "Assessment": "Generalized Anxiety Disorder (GAD)."}
    ),
    ("Psychiatry", "F32.1",
     "Honestly Doc, for the past six weeks it feels like I'm walking through waist-deep mud. Nothing brings me joy anymore—I haven't touched my guitar, I have zero appetite, and I wake up at 4 AM feeling guilty and worthless.",
     "Thank you for sharing that with me. Have you had any thoughts that life isn't worth living or any plans to harm yourself?",
     "No, I love my family and would never hurt myself, I just want to feel like a normal human being again.",
     "Mental status exam shows psychomotor slowing and constricted, depressed affect, but intact cognition and zero suicidal ideation. Your PHQ-9 score is 17 (moderately severe depression), and B12, ferritin, and TSH labs are normal. I recommend starting Sertraline 50mg daily alongside behavioral activation therapy.",
     "I saw St. John's Wort herbal pills at the health food store—can I just take those instead of prescription Sertraline?",
     "Over-the-counter St. John's Wort has inconsistent potency in moderately severe depression (PHQ-9 of 17) and causes dangerous liver enzyme drug interactions, whereas evidence-based Sertraline plus psychotherapy has proven remission rates. Follow up in 2 weeks.",
     {"walking through waist-deep mud": "psychomotor retardation and anergia", "nothing brings me joy anymore": "anhedonia", "wake up at 4 AM": "early morning terminal insomnia", "feeling guilty and worthless": "excessive inappropriate guilt and worthlessness"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Major Depressive Disorder (MDD), single episode, moderate-to-severe (PHQ-9 = 17), without suicidal ideation.",
      "Intervention": "Sertraline 50mg PO daily combined with structured Behavioral Activation psychotherapy.",
      "Comparison_Alternative": "Standardized SSRI therapy (Sertraline) + psychotherapy versus unregulated OTC herbal St. John's Wort monotherapy.",
      "Outcome": "Clinical remission (PHQ-9 <5), restoration of hedonic capacity, and normalization of sleep/appetite over 6–8 weeks."},
     {"CC": "Depressed mood, anhedonia, anergia, anorexia, and early morning awakening for 6 weeks.",
      "HPI": "Patient presents with 6-week history of pervasive low mood, marked anhedonia, low energy, poor appetite, terminal insomnia, and feelings of worthlessness (PHQ-9 = 17). Explicitly denies suicidal ideation.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Mild psychomotor slowing, depressed affect, goal-directed speech, no psychotic features. Normal TSH, Vitamin B12, and ferritin.",
      "Assessment": "Major Depressive Disorder (MDD), moderately severe."}
    ),
    ("Psychiatry", "F41.0",
     "Doc, four times this month I've been sitting at work or in the grocery store when out of nowhere a tidal wave of terror hits me—my heart hammers, my chest gets tight, my fingers tingle, and I'm convinced I'm dying.",
     "How long does that peak wave last, and are you starting to avoid going to the grocery store because you're afraid it'll happen again?",
     "Peaks in ten minutes and leaves me wiped out. And yeah, I started ordering groceries online because I'm terrified of having another meltdown in public.",
     "Resting ECG, troponin, thyroid panel, and physical exam are completely normal. You are experiencing Panic Disorder with early agoraphobic avoidance. I recommend Cognitive Behavioral Therapy with interoceptive exposure, paced diaphragmatic breathing, and daily Escitalopram.",
     "Should I just go to the emergency room for a heart workup every time my chest gets tight like that?",
     "Repeated ER cardiac workups reinforce the false belief that panic symptoms are heart attacks and worsen agoraphobia, whereas CBT exposure and SSRI therapy desensitize your fight-or-flight alarm system.",
     {"tidal wave of terror out of nowhere": "spontaneous recurrent panic attacks", "fingers tingle": "perioral/acral paresthesia secondary to hyperventilation", "terrified of having another meltdown in public": "anticipatory anxiety and agoraphobic avoidance"},
     ["Blood Pressure", "Temperature", "Medication Allergies", "Family History"],
     {"Patient_Diagnosis": "Panic Disorder with early agoraphobic avoidance.",
      "Intervention": "Panic-focused Cognitive Behavioral Therapy (interoceptive exposure) combined with daily SSRI pharmacotherapy (Escitalopram).",
      "Comparison_Alternative": "Outpatient CBT + SSRI maintenance therapy versus recurrent emergency department cardiac evaluations and avoidance behavior.",
      "Outcome": "Elimination of spontaneous panic attacks and resumption of independent public activities within 8 weeks."},
     {"CC": "Recurrent unexpected panic attacks (4 episodes this month) with anticipatory avoidance.",
      "HPI": "Patient describes abrupt surges of intense fear peaking within 10 minutes accompanied by palpitations, chest tightness, acral paresthesia, and fear of dying, leading to avoidance of grocery stores.",
      "Vitals": "HR 76 bpm, BP: Not Documented, Temp: Not Documented.",
      "Exam": "Normal cardiopulmonary exam, normal 12-lead ECG and TSH.",
      "Assessment": "Panic Disorder."}
    ),
    ("Psychiatry", "F51.01",
     "I haven't slept more than four hours a night in two months, Doc. I scroll on my phone in bed until midnight, then toss and turn looking at the clock until 3 AM worrying about how tired I'll be.",
     "Do you drink coffee in the afternoon, or work on your laptop in bed?",
     "I usually have an iced coffee at 4 PM, and yeah, I answer work emails in bed every night.",
     "Physical exam and sleep apnea screening (STOP-BANG score 1) are low risk. You have chronic primary psychophysiological insomnia driven by conditioned bedroom arousal and late caffeine. We will initiate Cognitive Behavioral Therapy for Insomnia (CBT-I)—specifically stimulus control, removing screens from the bedroom, and cutting caffeine after noon.",
     "Can you just prescribe me Zopiclone or Ambien sleeping pills to take every night?",
     "Hypnotic Z-drugs like Zopiclone cause rebound insomnia, tolerance, and grogginess without fixing poor sleep habits, whereas CBT-I is the gold-standard first-line treatment that restores natural deep sleep permanently.",
     {"toss and turn looking at the clock until 3 AM": "chronic sleep-onset insomnia with clock-watching arousal"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Chronic Primary Insomnia (Conditioned Psychophysiological Insomnia).",
      "Intervention": "Cognitive Behavioral Therapy for Insomnia (CBT-I: stimulus control, sleep restriction, screen removal, and afternoon caffeine cessation).",
      "Comparison_Alternative": "First-line non-pharmacological CBT-I protocol versus chronic nightly non-benzodiazepine hypnotic therapy (Zopiclone / Zolpidem).",
      "Outcome": "Reduction of sleep onset latency to <20 minutes and restoration of consolidated (>7 hours) sleep at 4 weeks."},
     {"CC": "Difficulty falling asleep and unrefreshing sleep (<4 hours/night) for 2 months.",
      "HPI": "Patient reports prolonged sleep onset latency associated with in-bed screen use, clock-watching, and 4 PM caffeine intake. Low STOP-BANG score (1).",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Alert, mild fatigue, normal oropharynx and cardiopulmonary exam.",
      "Assessment": "Chronic primary insomnia."}
    ),
    ("Psychiatry", "F90.0",
     "Doc, I'm really struggling in my first year of master's coursework—I read the same paragraph five times without absorbing a word, I lose my keys daily, and I constantly interrupt people without meaning to.",
     "Did teachers ever write on your elementary school report cards that you were daydreaming, restless, or disorganized as a kid?",
     "Every single report card from Grade 3 onward said 'bright student but constantly daydreams, loses homework, and blurts out answers'!",
     "Blood pressure is 118/74 mmHg, pulse 72 bpm. Your Adult ADHD Self-Report Scale (ASRS) is strongly positive across both inattentive and hyperactive-impulsive domains with confirmed childhood onset prior to age 12, and depression/anxiety screens are negative. You meet DSM-5 criteria for Adult ADHD. I recommend starting long-acting Methylphenidate (Concerta) 18mg each morning plus executive-function coaching.",
     "Can I just drink four energy drinks a day to focus instead of taking a prescription stimulant?",
     "High-dose caffeine energy drinks cause severe cardiovascular jitteriness and crashes without targeting prefrontal cortex dopamine/norepinephrine transporters, whereas extended-release Methylphenidate provides smooth, evidence-based 12-hour executive focus. Follow up in 3 weeks to check blood pressure and response.",
     {"read the same paragraph five times without absorbing a word": "sustained attention deficit and distractibility", "blurts out answers and interrupts": "verbal impulsivity"},
     ["Temperature", "Respiratory Rate", "Medication Allergies", "Family History"],
     {"Patient_Diagnosis": "Adult Attention-Deficit/Hyperactivity Disorder (ADHD), Combined Presentation.",
      "Intervention": "Extended-release Methylphenidate (Concerta) 18mg PO QAM combined with academic executive-function structuring.",
      "Comparison_Alternative": "Regulated extended-release stimulant pharmacotherapy (Methylphenidate) + behavioral structuring versus excessive high-caffeine energy drink self-medication.",
      "Outcome": "Improved sustained academic reading concentration, reduced organizational errors, and stable hemodynamic monitoring at 3 weeks."},
     {"CC": "Chronic academic inattention, disorganization, and verbal impulsivity.",
      "HPI": "Graduate student reports severe reading distractibility, frequent misplacement of items, and interrupting others, with documented symptom onset in elementary school (Grade 3). Positive ASRS screen.",
      "Vitals": "BP 118/74 mmHg, HR 72 bpm, Temp: Not Documented, RR: Not Documented.",
      "Exam": "Normal cardiovascular exam; linear thought process; negative GAD-7/PHQ-9.",
      "Assessment": "Adult ADHD, combined presentation."}
    ),

    # --- INFECTIOUS DISEASE / ENT (EXP_041 to EXP_045) ---
    ("Infectious Disease", "J02.0",
     "Doc, my throat feels like I'm swallowing broken glass since yesterday morning, I have a fever, and the glands in my neck are swollen and tender.",
     "Do you have a cough, runny nose, or hoarse voice?",
     "Zero cough and no runny nose at all, just the awful sore throat and fever.",
     "Temperature is 38.6°C, pulse 94 bpm. Exam shows beefy red tonsillar erythema with white tonsillar exudates and tender anterior cervical lymphadenopathy. Your Modified Centor score is 4 out of 4, and the in-office Rapid Strep Antigen test is positive. You have Group A Streptococcal pharyngitis. I'm prescribing Penicillin V 500mg twice daily for 10 days.",
     "Can I just gargle with warm salt water and stop taking the antibiotic once my throat feels better in two days?",
     "Salt water gargles help pain, but completing the full course of Penicillin V is essential to eradicate Group A Strep bacteria and prevent acute rheumatic fever and peritonsillar abscess.",
     {"swallowing broken glass": "severe odynophagia", "glands in my neck are swollen": "tender anterior cervical lymphadenopathy", "white spots on tonsils": "tonsillar exudates"},
     ["Blood Pressure", "Respiratory Rate", "Medication Allergies", "Smoking History"],
     {"Patient_Diagnosis": "Acute Group A Beta-Hemolytic Streptococcal (GAS) Pharyngitis (Centor Score 4, Rapid Strep Positive).",
      "Intervention": "Oral Penicillin V 500mg BID for 10 days plus acetaminophen/ibuprofen for analgesia.",
      "Comparison_Alternative": "Full 10-day targeted beta-lactam antibiotic regimen versus symptomatic salt-water gargles alone or premature antibiotic discontinuation.",
      "Outcome": "Complete eradication of pharyngeal Group A Streptococcus, resolution of odynophagia within 48 hours, and prevention of rheumatic fever."},
     {"CC": "Severe odynophagia, fever, and swollen neck glands for 24 hours without cough.",
      "HPI": "Patient presents with acute onset severe sore throat and fever. Denies cough, rhinorrhea, or hoarseness (Centor score = 4).",
      "Vitals": "Temp 38.6°C, HR 94 bpm, BP: Not Documented, RR: Not Documented.",
      "Exam": "Tonsillar erythema and exudates; tender anterior cervical lymphadenopathy; Rapid Strep Antigen positive.",
      "Assessment": "Acute Streptococcal Pharyngitis."}
    ),
    ("Infectious Disease", "J01.90",
     "I've had a stuffed-up nose and facial pressure under my eyes for four days now, and when I blow my nose the mucus is thick and green.",
     "Have you had a high fever over 39 degrees, double vision, or did you start getting better and then suddenly crash and get way worse?",
     "No fever at all, vision is fine, just been steadily congested for four days.",
     "Temperature is 36.8°C. Nasal mucosa is erythematous with purulent discharge, mild maxillary sinus tenderness, and normal extraocular movements. Because symptoms have lasted less than 10 days without 'double sickening' or high fever, this is Acute Viral Rhinosinusitis. I recommend intranasal Fluticasone spray, saline sinus irrigation, and analgesics.",
     "Wait, isn't green mucus a guarantee that it's a sinus infection that needs Amoxicillin?",
     "Green nasal mucus simply comes from normal white blood cell enzymes during a viral cold and does not indicate bacterial infection under 10 days, whereas saline irrigation and nasal steroid spray relieve sinus swelling without antibiotic side effects.",
     {"facial pressure under my eyes": "maxillary sinus pressure", "got better then suddenly crashed": "biphasic illness ('double sickening')"},
     ["Blood Pressure", "Heart Rate", "Respiratory Rate", "Medication Allergies"],
     {"Patient_Diagnosis": "Acute Viral Rhinosinusitis (duration <10 days without double-sickening).",
      "Intervention": "Intranasal corticosteroid spray (Fluticasone), isotonic saline nasal irrigation, and oral analgesics.",
      "Comparison_Alternative": "Symptomatic anti-inflammatory nasal therapy (antibiotic stewardship) versus empirical Amoxicillin triggered solely by nasal discharge color.",
      "Outcome": "Decongestion of ostiomeatal complex and complete symptom resolution within 7–10 days."},
     {"CC": "Nasal congestion, maxillary facial pressure, and purulent rhinorrhea for 4 days.",
      "HPI": "Patient reports 4 days of nasal congestion, maxillary pressure, and green rhinorrhea. Denies fever, visual changes, or biphasic worsening.",
      "Vitals": "Temp 36.8°C, BP: Not Documented, HR: Not Documented, RR: Not Documented.",
      "Exam": "Edematous nasal turbinates; mild bilateral maxillary tenderness; intact extraocular movements.",
      "Assessment": "Acute viral rhinosinusitis."}
    ),
    ("Infectious Disease", "A69.20",
     "Doc, ten days after I went hiking in the tall grass near Kenora, I noticed this growing red ring on the back of my left knee that looks exactly like a dartboard bullseye, and my joints ache all over.",
     "Did you ever spot a tick attached to your leg, or have you noticed any facial weakness or heart skipping?",
     "Pulled a tiny black-legged tick off that exact spot after the hike! No facial droop or heart skipping.",
     "Temperature is 37.6°C, pulse 80 bpm. Behind the left popliteal fossa is an 8 cm expanding annular erythematous macule with central clearing—classic Erythema Migrans. Because Erythema Migrans following a black-legged tick bite is diagnostic of early Lyme disease, we are starting oral Doxycycline 100mg twice daily for 14 days immediately.",
     "Should we wait two weeks for a Lyme blood test to come back positive before I start taking Doxycycline?",
     "Lyme antibody blood tests are frequently false-negative in the first two weeks of infection while the immune system is still mounting a response, whereas immediate Doxycycline treatment at the bullseye stage cures early Lyme disease and prevents Lyme carditis or arthritis.",
     {"red ring that looks like a dartboard bullseye": "erythema migrans (expanding annular erythematous lesion with central clearing)", "joints ache all over": "migratory arthralgia"},
     ["Blood Pressure", "Respiratory Rate", "Medication Allergies", "Past Medical History"],
     {"Patient_Diagnosis": "Early Localized Lyme Disease (Erythema Migrans) following Ixodes scapularis tick exposure.",
      "Intervention": "Immediate empirical Doxycycline 100mg PO BID for 14 days.",
      "Comparison_Alternative": "Immediate clinical diagnosis and Doxycycline initiation versus delaying treatment for early serologic testing (high false-negative window).",
      "Outcome": "Complete resolution of erythema migrans and prevention of disseminated neurologic/cardiac/articular Lyme disease."},
     {"CC": "Expanding bullseye rash on the left posterior knee and diffuse arthralgia 10 days post-tick bite.",
      "HPI": "Patient removed a tick after hiking in an endemic area 10 days ago and developed an expanding annular rash and arthralgias. Denies cranial neuropathy or palpitations.",
      "Vitals": "Temp 37.6°C, HR 80 bpm, BP: Not Documented, RR: Not Documented.",
      "Exam": "8 cm annular erythematous plaque with central clearing (Erythema Migrans) at left popliteal fossa; CN VII intact.",
      "Assessment": "Early localized Lyme disease."}
    ),
    ("Infectious Disease", "B27.90",
     "I've been wiped out for ten days, Doc. Worst sore throat of my life, low fever, and the sides of my neck look like golf balls. My roommate gave me leftover Amoxicillin two days ago and now I'm covered in a pink rash!",
     "Do you have any sharp pain in the upper left side of your belly under your ribs?",
     "Feels a little full on the left side of my stomach, yeah. And I'm sleeping fourteen hours a day.",
     "Temperature is 38.2°C. Exam shows exudative pharyngitis, prominent posterior cervical lymphadenopathy, mild splenomegaly on left upper quadrant palpation, and a diffuse maculopapular drug rash triggered by taking Amoxicillin during Epstein-Barr Virus infection. Monospot/EBV IgM test is positive for Infectious Mononucleosis. Stop the Amoxicillin immediately, rest, hydrate, and strictly avoid contact sports for 4 weeks.",
     "Does this pink rash mean I have a life-threatening penicillin allergy for the rest of my life?",
     "Over 80% of patients with viral Mononucleosis develop a temporary non-allergic rash when given Amoxicillin—it is a virus-drug interaction rather than true IgE penicillin anaphylaxis, and avoiding contact sports for 4 weeks is critical to prevent splenic rupture.",
     {"sides of my neck look like golf balls": "prominent bilateral posterior cervical lymphadenopathy", "full on the left side of my stomach": "splenomegaly"},
     ["Blood Pressure", "Heart Rate", "Respiratory Rate", "Family History"],
     {"Patient_Diagnosis": "Infectious Mononucleosis (Epstein-Barr Virus) complicated by secondary ampicillin/amoxicillin maculopapular exanthem and mild splenomegaly.",
      "Intervention": "Immediate cessation of Amoxicillin, supportive hydration/analgesia, and strict avoidance of contact sports/heavy lifting for 4 weeks.",
      "Comparison_Alternative": "Supportive viral management + splenic precautions (recognizing EBV-induced amoxicillin rash) versus mislabeling as permanent IgE penicillin allergy or continuing antibiotics.",
      "Outcome": "Resolution of pharyngitis/exanthem and prevention of traumatic splenic rupture over 4 weeks."},
     {"CC": "Severe sore throat, fatigue, neck swelling for 10 days, and generalized rash after taking Amoxicillin.",
      "HPI": "Patient reports 10 days of fever, severe pharyngitis, profound hypersomnia, and neck swelling; developed maculopapular rash 48 hours after taking non-prescribed Amoxicillin.",
      "Vitals": "Temp 38.2°C, BP: Not Documented, HR: Not Documented, RR: Not Documented.",
      "Exam": "Exudative tonsillitis, posterior cervical lymphadenopathy, palpable splenic tip in LUQ, diffuse morbilliform rash. Monospot positive.",
      "Assessment": "Infectious Mononucleosis with EBV-associated amoxicillin exanthem."}
    ),
    ("Infectious Disease", "H60.331",
     "Ever since I spent the weekend swimming at the community pool, my right ear itches like crazy, hurts like fire if I tug on my earlobe, and has cloudy fluid oozing out.",
     "Have you been using cotton Q-tips inside your ear canal, or do you have a fever?",
     "Yeah, I tried drying it out deep inside with Q-tips on Sunday. No fever.",
     "Temperature is 36.6°C. Pulling on your right pinna and pushing the tragus causes severe pain. Otoscopy shows an edematous, erythematous right external auditory canal with moist debris and an intact tympanic membrane. You have acute Otitis Externa (swimmer's ear). Keep the ear dry, stop Q-tips, and use Ciprofloxacin-Dexamethasone ototopical drops twice daily for 7 days.",
     "Wouldn't oral antibiotic pills work better than ear drops to clear the ear infection?",
     "Topical Ciprofloxacin-Dexamethasone drops deliver a thousand times higher antibiotic concentration directly to the outer ear canal skin and reduce swelling faster than oral antibiotics, which are unnecessary for uncomplicated swimmer's ear.",
     {"hurts like fire if I tug on my earlobe": "pronounced tragal and pinna traction tenderness", "cloudy fluid oozing out": "purulent otorrhea"},
     ["Blood Pressure", "Heart Rate", "Respiratory Rate", "Medication Allergies"],
     {"Patient_Diagnosis": "Right Acute Diffuse Otitis Externa ('Swimmer's Ear').",
      "Intervention": "Topical Ciprofloxacin 0.3% / Dexamethasone 0.1% otic suspension 4 drops BID x 7 days, dry-ear precautions, and cessation of cotton-swab instrumentation.",
      "Comparison_Alternative": "Targeted ototopical fluoroquinolone-steroid drops versus systemic oral antibiotic therapy for uncomplicated otitis externa.",
      "Outcome": "Resolution of canal edema, otalgia, and otorrhea within 7 days."},
     {"CC": "Right ear pain aggravated by earlobe traction, pruritus, and otorrhea for 3 days.",
      "HPI": "Symptoms began after swimming and cotton-swab use. Reports severe right otalgia with pinna movement and cloudy discharge. Denies fever.",
      "Vitals": "Temp 36.6°C, BP: Not Documented, HR: Not Documented, RR: Not Documented.",
      "Exam": "Positive right pinna traction and tragal tenderness; erythematous, edematous right external auditory canal with exudate; intact tympanic membrane.",
      "Assessment": "Right acute otitis externa."}
    ),

    # --- NEPHROLOGY & UROLOGY (EXP_046 to EXP_050) ---
    ("Nephrology/Urology", "N39.0",
     "Doc, since yesterday morning it feels like I'm peeing razor blades, and I feel like I have to run to the washroom every fifteen minutes just to pass three drops.",
     "Do you have any fever, shaking chills, or pain in your back under your ribs?",
     "No fever or chills, and my back doesn't hurt at all, just cramps low down over my bladder.",
     "Temperature is 36.7°C. Suprapubic palpation shows mild tenderness, and there is zero costovertebral angle (CVA) flank tenderness. Urine dipstick is positive for both leukocyte esterase and nitrites. You have an uncomplicated acute cystitis (UTI). I'm prescribing Nitrofurantoin (MacroBID) 100mg twice daily for 5 days.",
     "Can I just drink two jugs of cranberry juice over the weekend instead of taking antibiotics?",
     "Cranberry products may slightly reduce future recurrent infections, but once active bacterial cystitis with positive nitrites is established, Nitrofurantoin 100mg for 5 days clears the infection rapidly and prevents it from climbing to your kidneys.",
     {"peeing razor blades": "severe dysuria", "run to the washroom every fifteen minutes for three drops": "urinary frequency and urgency", "cramps low down over my bladder": "suprapubic discomfort"},
     ["Blood Pressure", "Heart Rate", "Medication Allergies", "Pregnancy Status"],
     {"Patient_Diagnosis": "Acute Uncomplicated Bacterial Cystitis (Lower Urinary Tract Infection).",
      "Intervention": "Nitrofurantoin monohydrate/macrocrystals (MacroBID) 100mg PO BID for 5 days plus oral hydration.",
      "Comparison_Alternative": "First-line 5-day targeted Nitrofurantoin antibiotic therapy versus cranberry juice monotherapy during active infection.",
      "Outcome": "Eradication of bacteriuria, resolution of dysuria/frequency within 48 hours, and prevention of ascending pyelonephritis."},
     {"CC": "Severe dysuria, urinary frequency, urgency, and suprapubic discomfort for 24 hours.",
      "HPI": "Patient reports acute onset dysuria ('razor blades'), frequency every 15 minutes, urgency, and suprapubic cramping. Denies fever, rigors, or flank pain.",
      "Vitals": "Temp 36.7°C, BP: Not Documented, HR: Not Documented.",
      "Exam": "Mild suprapubic tenderness; negative bilateral CVA tenderness. Urinalysis positive for nitrites and leukocyte esterase.",
      "Assessment": "Acute uncomplicated cystitis (UTI)."}
    ),
    ("Nephrology/Urology", "N20.0",
     "I was in the middle of breakfast two hours ago when I got hit with the worst pain of my life in my right back that wraps right around into my groin—it comes in waves and I can't sit still!",
     "Did your urine look pink or red this morning, and do you have a fever?",
     "Yeah, my pee looked like pink cranberry juice an hour ago! No fever, just nauseous from the pain.",
     "Temperature is 36.9°C, BP 150/92 mmHg, pulse 98 bpm. Right costovertebral angle tenderness is present, abdomen is soft, and urinalysis shows microscopic hematuria without nitrites or leukocytes. Non-contrast CT KUB shows a 4 mm distal right ureterovesical junction (UVJ) calculus without hydronephrosis or infection. We will give Ketorolac/Ibuprofen for pain, Tamsulosin 0.4mg daily as medical expulsive therapy, and a urine strainer.",
     "Do I need emergency laser surgery today to blast the stone out?",
     "Stones under 5 mm in the distal ureter pass spontaneously over 80% of the time, and adding an alpha-blocker like Tamsulosin relaxes the ureter muscle to speed passage safely without invasive ureteroscopy surgery. Return immediately if you develop fever.",
     {"pain in my right back that wraps around into my groin": "acute right renal/ureteral colic radiating to the ipsilateral groin", "pee looked like pink cranberry juice": "gross hematuria"},
     ["Respiratory Rate", "Medication Allergies", "Past Surgical History", "Family History"],
     {"Patient_Diagnosis": "Acute Right Distal Ureterolithiasis (4 mm UVJ calculus) without sepsis or high-grade obstruction.",
      "Intervention": "Medical Expulsive Therapy (Tamsulosin 0.4mg PO daily), NSAID analgesia, high oral fluid intake, and urine straining.",
      "Comparison_Alternative": "Conservative Medical Expulsive Therapy (Tamsulosin + NSAIDs) for a 4 mm distal stone versus immediate invasive ureteroscopic laser lithotripsy.",
      "Outcome": "Spontaneous passage of the 4 mm calculus within 7–14 days and complete pain resolution."},
     {"CC": "Acute severe colicky right flank pain radiating to the groin and gross hematuria for 2 hours.",
      "HPI": "Patient presents with sudden paroxysmal right flank-to-groin pain, restlessness, nausea, and gross hematuria. Denies fever or chills.",
      "Vitals": "Temp 36.9°C, BP 150/92 mmHg, HR 98 bpm, RR: Not Documented.",
      "Exam": "Right CVA tenderness; soft abdomen. Urinalysis positive for blood, negative for infection; CT KUB shows 4 mm right UVJ calculus.",
      "Assessment": "Acute right ureterolithiasis (4 mm distal calculus)."}
    ),
    ("Nephrology/Urology", "N40.1",
     "Doc, over the past year my pee stream has gotten super weak—it takes forever to get started, dribbles at the end, and I'm waking up three times every night feeling like my bladder isn't empty.",
     "Have you ever seen blood in your urine, had burning when you pee, or been completely unable to urinate?",
     "No blood or burning, and it always comes out eventually, just takes a minute of standing there.",
     "Blood pressure is 128/78 mmHg. Digital rectal exam shows a symmetrically enlarged, smooth, rubbery prostate without nodules, urinalysis is completely clean, PSA is normal at 1.8 ng/mL, and post-void residual bladder scan is 65 mL. You have Benign Prostatic Hyperplasia (BPH) with moderate lower urinary tract symptoms. I recommend starting Tamsulosin 0.4mg daily at bedtime.",
     "Can I just take over-the-counter Saw Palmetto berry capsules instead of prescription Tamsulosin?",
     "Large clinical trials show Saw Palmetto is no better than placebo for improving urine flow or nighttime waking, whereas Tamsulosin directly relaxes smooth muscle at the prostate and bladder neck within days. Let's review in 4 weeks.",
     {"takes forever to get started": "urinary hesitancy", "pee stream has gotten super weak and dribbles": "decreased urinary stream force and terminal dribbling", "feeling like my bladder isn't empty": "sensation of incomplete bladder emptying"},
     ["Heart Rate", "Temperature", "Medication Allergies", "Smoking History"],
     {"Patient_Diagnosis": "Symptomatic Benign Prostatic Hyperplasia (BPH) with moderate Lower Urinary Tract Symptoms (LUTS).",
      "Intervention": "Tamsulosin 0.4mg PO daily (taken 30 minutes after the same meal) plus evening fluid moderation.",
      "Comparison_Alternative": "Alpha-1a adrenergic blocker pharmacotherapy (Tamsulosin) versus OTC herbal phytotherapy (Saw Palmetto).",
      "Outcome": "Improvement in peak urinary flow, reduction in nocturia frequency, and reduced IPSS symptom score at 4 weeks."},
     {"CC": "Progressive urinary hesitancy, weak stream, terminal dribbling, and nocturia (3x/night) for 1 year.",
      "HPI": "Patient reports obstructive and irritative LUTS including hesitancy, weak stream, incomplete emptying sensation, and nocturia. Denies hematuria or dysuria.",
      "Vitals": "BP 128/78 mmHg, HR: Not Documented, Temp: Not Documented.",
      "Exam": "DRE: smooth, symmetrically enlarged prostate without nodularity; UA negative; PSA 1.8 ng/mL; PVR 65 mL.",
      "Assessment": "Benign Prostatic Hyperplasia (BPH) with LUTS."}
    ),
    ("Nephrology/Urology", "N18.30",
     "I came in to go over my routine diabetes and blood pressure lab work from Tuesday—the portal flagged my kidney numbers and urine protein in red.",
     "Have you noticed any foamy urine in the toilet bowl, ankle swelling, or been taking over-the-counter ibuprofen?",
     "My urine does look pretty bubbly and foamy lately. No ankle swelling, and no ibuprofen.",
     "Blood pressure is 138/84 mmHg. Your eGFR is 52 mL/min/1.73m2 (stable over 4 months) and your urine Albumin-to-Creatinine Ratio (uACR) is elevated at 42 mg/mmol. You have Stage 3a Diabetic Chronic Kidney Disease with albuminuria. To protect your kidney filters, we are adding an SGLT2 inhibitor—Empagliflozin 10mg daily—on top of your current ACE inhibitor.",
     "Wait, if my blood sugar is already decent on Metformin, why add another diabetes pill like Empagliflozin?",
     "Empagliflozin isn't just for blood sugar—it directly lowers pressure inside the kidney glomeruli, reduces foamy urine protein loss, and slows CKD progression by nearly 40% beyond what Metformin and blood pressure pills can do alone.",
     {"bubbly and foamy urine": "proteinuria / albuminuria"},
     ["Heart Rate", "Temperature", "Respiratory Rate", "Medication Allergies"],
     {"Patient_Diagnosis": "Stage 3a Chronic Kidney Disease (eGFR 52 mL/min) with moderate-to-severe albuminuria (uACR 42 mg/mmol) secondary to Diabetic Nephropathy.",
      "Intervention": "Initiation of SGLT2 inhibitor therapy (Empagliflozin 10mg PO daily) alongside existing RAAS blockade.",
      "Comparison_Alternative": "Dual nephroprotective therapy (ACE inhibitor + SGLT2 inhibitor Empagliflozin) versus continuing Metformin and ACE inhibitor without SGLT2 inhibition.",
      "Outcome": "Reduction in urine Albumin-to-Creatinine Ratio (uACR) and preservation of long-term eGFR slope at 3-month follow-up."},
     {"CC": "Abnormal renal function and elevated urine albumin-to-creatinine ratio on routine lab review.",
      "HPI": "Patient with Type 2 diabetes and hypertension presents for lab review; notes foamy urine. Denies pedal edema or NSAID use.",
      "Vitals": "BP 138/84 mmHg, HR: Not Documented, Temp: Not Documented, RR: Not Documented.",
      "Exam": "No peripheral edema; clear lungs. Labs: eGFR 52 mL/min/1.73m2, uACR 42 mg/mmol.",
      "Assessment": "Stage 3a Diabetic Chronic Kidney Disease with albuminuria."}
    ),
    ("Nephrology/Urology", "N39.41",
     "Doc, this is really awkward to talk about, but for six months whenever I hear running water or pull into my driveway, I get this sudden overwhelming urge to pee and sometimes leak before I can unlock the door.",
     "Does it also leak when you cough, sneeze, or laugh, or is it strictly with that sudden urge?",
     "Not when I cough or sneeze at all—only when that sudden panic urge hits me.",
     "Pelvic and abdominal exam are normal, post-void residual volume is 30 mL, and urinalysis is negative for infection. You have Overactive Bladder (OAB) with urge urinary incontinence. First-line management is timed bladder retraining, pelvic floor Kegel exercises, and cutting back on evening tea, paired with Mirabegron 25mg daily if needed.",
     "My sister had a surgical mesh sling put in for bladder leaking—should I get that sling surgery too?",
     "Mid-urethral mesh slings only treat stress incontinence caused by coughing or sneezing and can actually worsen urge incontinence, whereas bladder retraining and beta-3 agonist therapy (Mirabegron) specifically calm overactive detrusor bladder spasms.",
     {"sudden overwhelming urge to pee and leak before unlocking the door": "urgency urinary incontinence ('latchkey incontinence')"},
     ["Blood Pressure", "Heart Rate", "Temperature", "Medication Allergies"],
     {"Patient_Diagnosis": "Overactive Bladder (OAB) syndrome with Urge Urinary Incontinence.",
      "Intervention": "Behavioral bladder retraining, pelvic floor muscle therapy (Kegels), fluid/caffeine modification, and Mirabegron 25mg PO daily.",
      "Comparison_Alternative": "Targeted detrusor rehabilitation + beta-3 adrenergic agonist therapy versus surgical mid-urethral mesh sling (indicated only for stress incontinence).",
      "Outcome": "Reduction in urgency incontinence episodes and increased inter-voiding intervals at 6-week follow-up."},
     {"CC": "Sudden urinary urgency with urge incontinence episodes for 6 months.",
      "HPI": "Patient reports sudden compelling urinary urgency triggered by environmental cues (running water, arriving home) with urge leakage. Denies stress incontinence with coughing/sneezing.",
      "Vitals": "BP: Not Documented, HR: Not Documented, Temp: Not Documented.",
      "Exam": "Normal abdominal/pelvic exam; negative urinalysis; Post-Void Residual (PVR) = 30 mL.",
      "Assessment": "Overactive Bladder (OAB) with urge urinary incontinence."}
    )
]

# 3. Assemble the 50 Complete Records by combining the 50 Small-Talk pairs with the 50 Clinical Cases
custom_experimental_records = []

for idx in range(50):
    case_id = f"EXP_{idx + 1:03d}"
    doc_banter, pat_banter, banter_summary = SMALL_TALK_POOL[idx]
    (specialty, icd10, pat_complaint, doc_q, pat_followup,
     doc_exam_plan, pat_alt_q, doc_pico_reply,
     ling_shifts, omitted_list, pico_dict, soap_dict) = CLINICAL_CASES_SPEC[idx]

    # Build the multi-turn noisy dialogue
    full_dialogue = (
        f"Doctor: {doc_banter}\n"
        f"Patient: {pat_banter} {pat_complaint}\n"
        f"Doctor: {doc_q}\n"
        f"Patient: {pat_followup}\n"
        f"Doctor: {doc_exam_plan}\n"
        f"Patient: {pat_alt_q}\n"
        f"Doctor: {doc_pico_reply}"
    )

    # Dynamically format the omitted fields as explicit "Not Documented" lines in the reference SOAP
    omitted_soap_lines = "\n".join([f"- {field}: Not Documented." for field in omitted_list])

    # Build the gold-standard clinician Reference SOAP note incorporating PICO & Negative Guardrails
    reference_soap = (
        f"Subjective:\n"
        f"- Chief Complaint: {soap_dict['CC']}\n"
        f"- HPI: {soap_dict['HPI']}\n"
        f"{omitted_soap_lines}\n\n"
        f"Objective:\n"
        f"- Vitals: {soap_dict['Vitals']}\n"
        f"- Physical Exam & Labs: {soap_dict['Exam']}\n\n"
        f"Assessment:\n"
        f"- {soap_dict['Assessment']}\n\n"
        f"Plan:\n"
        f"- PICO Summary:\n"
        f"  * [P] Patient/Diagnosis: {pico_dict['Patient_Diagnosis']}\n"
        f"  * [I] Intervention: {pico_dict['Intervention']}\n"
        f"  * [C] Comparison/Alternative: {pico_dict['Comparison_Alternative']}\n"
        f"  * [O] Outcome: {pico_dict['Outcome']}"
    )

    custom_experimental_records.append({
        "id": case_id,
        "clinical_domain": specialty,
        "icd10_code": icd10,
        "dialogue": full_dialogue,
        "small_talk_present": banter_summary,
        "linguistic_shifts": ling_shifts,
        "omitted_fields": omitted_list,
        "pico_ground_truth": pico_dict,
        "reference_soap": reference_soap
    })

# 4. Convert to DataFrame and serialize to JSON and Parquet
df_custom_50 = pd.DataFrame(custom_experimental_records)

json_path = os.path.join(OUTPUT_DIR, "custom_experimental_50_cases.json")
parquet_path = os.path.join(OUTPUT_DIR, "custom_experimental_50_cases.parquet")

df_custom_50.to_json(json_path, orient="records", indent=2)
df_custom_50.to_parquet(parquet_path, index=False)

print("=" * 70)
print(f"[SUCCESS] Generated & Saved {len(df_custom_50)} Controlled Experimental Stress-Test Cases!")
print(f"  -> JSON Path   : {json_path}")
print(f"  -> Parquet Path: {parquet_path}")
print("=" * 70)

## 5. Dataset Quality Audit & Domain Distribution
Before moving to **Notebook 02 (Semantic Few-Shot Retriever)** and **Notebook 03 (IBM Granite 3.1 Before-vs-After Pipeline)**, we run a quick verification audit on our 50-case stress-test corpus to verify:
1. Balanced distribution across all **10 clinical specialties** (5 cases per specialty).
2. Total count of annotated **Linguistic Shift pairs** and **Omitted Clinical Variables** (Negative Guardrail triggers).
3. Inspection of a complete sample record (`EXP_001`).

In [ ]:
# ==============================================================================
# AUDIT & INSPECT THE 50-CASE CUSTOM EXPERIMENTAL CORPUS
# Purpose: Generate summary statistics for the IEEE Paper Methodology section
#          and inspect a full sample record.
# ==============================================================================

# 1. Compute summary counts across the 50 cases
total_cases = len(df_custom_50)
total_ling_shifts = sum(len(x) for x in df_custom_50["linguistic_shifts"])
total_omitted_triggers = sum(len(x) for x in df_custom_50["omitted_fields"])

print(f"Total Custom Experimental Dialogues : {total_cases}")
print(f"Total Linguistic Shift Ground-Truths: {total_ling_shifts} colloquial-to-clinical pairs")
print(f"Total Negative Guardrail Triggers   : {total_omitted_triggers} omitted clinical fields")
print("\nBreakdown by Clinical Specialty:")
print(df_custom_50["clinical_domain"].value_counts().to_string())

# 2. Display a clean preview table of the first 10 cases
display(df_custom_50[["id", "clinical_domain", "icd10_code", "small_talk_present", "omitted_fields"]].head(10))

# 3. Print full inspection of Case EXP_001
print("\n" + "=" * 70)
print("SAMPLE RECORD INSPECTION: EXP_001")
print("=" * 70)
sample = df_custom_50.iloc[0]
print(f"[DIALOGUE]\n{sample['dialogue']}\n")
print(f"[GROUND TRUTH LINGUISTIC SHIFTS]\n{json.dumps(sample['linguistic_shifts'], indent=2)}\n")
print(f"[GROUND TRUTH OMITTED FIELDS (MUST OUTPUT 'Not Documented')]\n{sample['omitted_fields']}\n")
print(f"[REFERENCE SOAP NOTE WITH PICO]\n{sample['reference_soap']}")